# Advanced Techniques for Language Models

**Mini-Assignment 2**

---

António Cruz (140129), Bruno Santos (140586), Pedro Miranda (129268)


# 1. Introduction

Mini-Assignment 1 produced a small language model adapted to the IT job-postings domain by continued pretraining on raw postings. That model is a text completer: given a posting, it can continue it. What it cannot do is take a recruiter request like "we need a backend engineer with Django and PostgreSQL experience" and produce a properly structured job posting from a sentence-long brief.

Mini-Assignment 2 is the alignment stage that makes the model do that. It has two sub-stages. Supervised fine-tuning teaches the model to recognise a recruiter request and produce a structured response. Preference alignment via Direct Preference Optimization then refines those responses against a written rubric, using preferences ranked by a separate local LLM acting as judge.

The report walks the full pipeline and ends with a comparison of six model states on a fixed set of twenty held-out recruiter prompts: the untrained base, the supervised fine-tuned model, and four DPO-aligned variants trained at different KL-coefficient values. The Mini-Assignment 1 report covers the previous stage; the Final Project will integrate this aligned model into a downstream system.


## 1.1 Pipeline overview

The full path from a pretrained base to a deployment-ready aligned model has four stages. Mini-Assignment 2 covers the last two.

1. Pretraining. The base model is trained on general text by its authors. We start from SmolLM2-360M, an open-source decoder under one billion parameters.
2. Continued pretraining (Mini-Assignment 1). The base is adapted to the job-postings domain by continued next-token training on raw IT job descriptions. The result is a text completer fluent in the domain but not an instruction follower.
3. Supervised fine-tuning (this report). The domain-adapted model learns to recognise recruiter instructions and produce structured responses, by training on roughly seven thousand (query, structured posting) pairs.
4. Preference alignment (this report). The supervised model is refined to prefer outputs that respect the rubric (faithfulness to the request, four required sections, professional language, no repetition), using preferences ranked by a separate local LLM.

Three external LLMs participate so that the data-preparation and judging roles stay independent of the model being aligned. Gemma 4 E2B served as the Mini-Assignment 1 ETL teacher (the `atlm_teacher` agent), producing the (query, posting) pairs that supervised fine-tuning trains on; Gemma does not act as judge in any stage. Nemotron Nano 4B is the RLAIF listwise judge in Section 4.3, via the `atlm_rlaif_judge` preset on `agent_server`. Granite 3.3 2B is the pairwise eval judge in Section 6.4, via the `atlm_eval_judge` preset. Both judge presets ship from [`documents/development/agent_server_setup/`](../../documents/development/agent_server_setup/); the calibration battery that picked them from six candidates is in [`documents/development/llm_models_performance.md`](../../documents/development/llm_models_performance.md), and the production-scale judge ceiling the calibration did not surface is the subject of Section 7.3.

Each stage adds a distinct capability: pretraining gives general language, continued pretraining gives domain fluency, supervised fine-tuning gives instruction-following, and preference alignment shapes the quality of those instruction-following outputs.


# 2. Starting point: the Mini-Assignment 1 checkpoint

Mini-Assignment 1 produced two trained variants of SmolLM2-360M: a full fine-tune and a LoRA adapter. The Mini-Assignment 1 report selected the LoRA variant as the best in-domain fit, and that is the checkpoint carried into this work.

Before any further training, the LoRA adapter is merged into the base. The result is a single self-contained checkpoint, equivalent at inference time to (base + Mini-Assignment 1 LoRA) but with no PEFT plumbing. Every subsequent stage of this notebook trains a fresh LoRA on top of a plain base, with no stacked-adapter bookkeeping.


## 2.1 Merge the Mini-Assignment 1 LoRA into the base

Load the base, attach the Mini-Assignment 1 LoRA, call `merge_and_unload`, save the result. The operation runs on CPU in a few seconds. The base model id is read from the adapter config rather than hardcoded, so the cell does not need editing if the underlying base ever changes.


In [1]:
# Merge the Mini-Assignment 1 LoRA into the SmolLM2-360M base.
# This runs on CPU: from_pretrained loads the weights to CPU and no tensors
# are moved to the GPU, so the GPU stays free for the training cells below.
# The merged model is saved to outputs/mp1-360m/merged/ and is the starting
# point for the supervised fine-tuning and DPO stages below.
import json
import time
import gc
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Locate the project root: walk up to a folder containing data/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

LORA_DIR   = PROJECT_ROOT / "outputs" / "mp1-360m" / "lora"
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

# Read the base model id from the adapter config; never hardcoded.
adapter_cfg = json.loads((LORA_DIR / "adapter_config.json").read_text())
BASE_MODEL = adapter_cfg["base_model_name_or_path"]
print(f"Base model:    {BASE_MODEL}")
print(f"LoRA adapter:  {LORA_DIR}")
print(f"Merged output: {MERGED_DIR}")

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, str(LORA_DIR), torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Free CPU memory before the rest of the notebook loads the model again.
del model, tokenizer
gc.collect()

print(f"Merged in {time.time() - t0:.1f}s.")


Base model:    HuggingFaceTB/SmolLM2-360M
LoRA adapter:  /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/lora
Merged output: /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged in 5.2s.


## 2.2 Load the merged model

With the merged checkpoint on disk, the rest of the notebook loads the model from `outputs/mp1-360m/merged/`. From here onwards the model is a regular pretrained-style checkpoint; downstream stages have no need to know that a Mini-Assignment 1 LoRA was ever involved.


In [2]:
# Load the merged Mini-Assignment 1 model for the rest of the notebook.
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

ma1_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1_model     = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
total_params  = sum(p.numel() for p in ma1_model.parameters())

print(f"Loaded merged MA1 model from {MERGED_DIR}")
print(f"Total parameters: {total_params:,}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded merged MA1 model from /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged
Total parameters: 361,821,120


# 3. Supervised fine-tuning

A continued-pretraining checkpoint is a text completer, not an instruction follower. DPO and other alignment techniques assume the policy already follows instructions, so a short supervised pass is needed before alignment.

This section trains the merged Mini-Assignment 1 model on roughly seven thousand (recruiter query, structured posting) pairs produced by the `atlm_teacher` ETL agent (Gemma 4). The result is a domain-adapted, instruction-following checkpoint ready for the preference-alignment stage in Section 5.


## 3.1 Prompt template

Mini-Assignment 1 left the model as a domain-fluent text completer that has never seen an instruction. Before alignment can happen, the model has to learn where a request ends, where a response begins, and what shape that response should take. A consistent prompt template carries that signal during supervised fine-tuning.

The template is Alpaca-inspired: a one-sentence system framing plus two delimited fields.

```
You are a recruitment assistant. Given a brief recruiter request, write a complete structured job posting in Markdown.

### Request
{query}

### Posting
{jd}
```

At inference time the prompt the model sees is everything up to and including `### Posting\n`; the model writes the posting from there.

Three reasons for this template, against the leaner key-value and ChatML alternatives considered:

1. Explicit task framing. SmolLM2-360M is small and has never been instruction-tuned. A one-sentence preamble gives the model an unambiguous anchor for the task, which is worth its roughly 25-token cost.
2. No clash with response content. The structured postings in the training data already use `##` headings (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`). The template uses `###` separators, so the model can distinguish a template marker from a response heading.
3. No special-token machinery. Plain-text separators tokenise into existing vocabulary; no new embeddings to learn, no chat template to register on the tokenizer. The HuggingFace TRL `SFTTrainer` consumes the template through a simple formatting function.

Every one of the roughly 7,500 training pairs is wrapped in this template.


## 3.2 Training data

The supervised fine-tuning data are the 2,507 records in `data/processed/converted.jsonl`, the output of the `atlm_teacher` ETL agent that took raw Djinni postings and produced clean, structured job descriptions in Markdown.

Each record carries three independent recruiter queries pointing to the same job description. The three queries are fanned out into independent training examples so the model sees the same target paired with three different phrasings of the same underlying request. This is a cheap form of augmentation that teaches the model to be robust to phrasing variation, and it roughly triples the effective dataset size.

The records are split 90/10 into train and validation at the record level, not at the query level, so no job description appears in both splits. Seed 42 makes the split reproducible.


In [3]:
# Load the SFT data, fan out the three queries per record, format with the
# Section 3.1 template, and split into train/validation at the record level.
import json
import random
from pathlib import Path

from datasets import Dataset

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
SEED = 42

# Read raw records (each has `queries` (list of 3) and `job_description`).
records = [json.loads(ln) for ln in open(SFT_DATA, encoding="utf-8")]
print(f"Loaded {len(records):,} records from {SFT_DATA.name}")

# Split at the record level (not the query level), so no JD appears in both
# train and val. 90/10 split, seed 42.
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records   = records[:n_val]
train_records = records[n_val:]
print(f"Split: {len(train_records):,} train records, {len(val_records):,} val records")

# Prompt template, exactly as described in Section 3.1.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def format_example(query: str, jd: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n{jd}"
    )

def fan_out(rs):
    out = []
    for r in rs:
        for q in r["queries"]:
            out.append({"text": format_example(q, r["job_description"])})
    return out

train_examples = fan_out(train_records)
val_examples   = fan_out(val_records)
print(f"After fan-out: {len(train_examples):,} train examples, "
      f"{len(val_examples):,} val examples")

sft_train = Dataset.from_list(train_examples)
sft_val   = Dataset.from_list(val_examples)

# A quick look at one formatted training example, truncated for readability.
print()
print("Sample formatted example:")
print("-" * 60)
print(train_examples[0]["text"][:800])
print("...")


Loaded 2,507 records from converted.jsonl
Split: 2,257 train records, 250 val records
After fan-out: 6,771 train examples, 750 val examples

Sample formatted example:
------------------------------------------------------------
You are a recruitment assistant. Given a brief recruiter request, write a complete structured job posting in Markdown.

### Request
We need someone to help us ensure our product is high quality before it gets to the customer.

### Posting
# Manual Test Engineer

## Summary
We are looking for a motivated and technically skilled Test Engineer to join an international team. The primary focus of this role is ensuring the successful delivery of a high-quality product through rigorous testing and framework development.

## Required Skills
- Test case writing and execution
- Test framework development and maintenance
- Integration with CI/CD infrastructure
- Problem-solving skills
- Technical aptitude

## Responsibilities
- Write and execute tests on a daily basis to e

## 3.3 Training configuration

Supervised fine-tuning continues the parameter-efficient approach from Mini-Assignment 1: a fresh LoRA adapter trained on top of the merged base, leaving the base weights untouched. This keeps both disk footprint and GPU memory small, and matches the LoRA shape (r=16, alpha=32, dropout=0.05, attention plus feed-forward projections) that worked in the previous stage.

Hyperparameters mirror Mini-Assignment 1 where applicable: bf16 precision, micro-batch 4 with gradient accumulation 4 (effective batch 16), warmup ratio 0.03, weight decay 0.01, seed 42. Maximum sequence length is 1024 tokens, comfortably above the longest (query, posting) pair. Training runs for two epochs over the roughly 6,800 training examples; with the fanned-out queries, each unique job description is seen six times in total (three phrasings, two epochs).

The learning rate is 2e-4, the conventional value for LoRA. The loss is computed over the entire formatted sequence (preamble plus request plus response) rather than only the response. For a model this small the inefficiency is minor and the configuration stays simpler.

The trained adapter, log history and a summary file go to `outputs/ma2-360m-sft/`. The run tag is `ma2-360m-sft`.


In [4]:
# Training configuration for the SFT run; consumed by run_sft() below.
SFT_RUN = "ma2-360m-sft"
SFT_OUT = PROJECT_ROOT / "outputs" / SFT_RUN

SFT_CFG = {
    "epochs": 2,
    "per_device_batch": 4,
    "grad_accum": 4,                   # effective batch = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_seq_length": 1024,
    # LoRA shape, same as Mini-Assignment 1.
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "seed": SEED,
}
print("SFT config:")
for k, v in SFT_CFG.items():
    print(f"  {k}: {v}")
print(f"output dir: {SFT_OUT}")


SFT config:
  epochs: 2
  per_device_batch: 4
  grad_accum: 4
  learning_rate: 0.0002
  warmup_ratio: 0.03
  weight_decay: 0.01
  max_seq_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  seed: 42
output dir: /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft


## 3.4 Running supervised fine-tuning

`run_sft` wraps the merged Mini-Assignment 1 model in a fresh LoRA, configures the HuggingFace TRL `SFTTrainer`, runs training, and saves the adapter plus a small summary (trainable parameter count, wall-clock minutes, final validation loss and perplexity). It mirrors the shape of `run_training` from Mini-Assignment 1.

The cell needs a CUDA GPU with bf16 support. On the RTX 4090 used here it takes roughly 10 to 15 minutes.


In [5]:
# Defines run_sft(): one supervised fine-tuning run on top of the merged
# base, producing a new LoRA adapter at outputs/ma2-360m-sft/.
import json
import math
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def run_sft():
    set_seed(SFT_CFG["seed"])
    SFT_OUT.mkdir(parents=True, exist_ok=True)

    # Load the merged Mini-Assignment 1 model (the SFT starting point).
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=torch.bfloat16
    )

    peft_cfg = LoraConfig(
        r=SFT_CFG["lora_r"],
        lora_alpha=SFT_CFG["lora_alpha"],
        lora_dropout=SFT_CFG["lora_dropout"],
        target_modules=SFT_CFG["lora_target_modules"],
        task_type="CAUSAL_LM",
    )

    args = SFTConfig(
        output_dir=str(SFT_OUT),
        num_train_epochs=SFT_CFG["epochs"],
        per_device_train_batch_size=SFT_CFG["per_device_batch"],
        per_device_eval_batch_size=SFT_CFG["per_device_batch"],
        gradient_accumulation_steps=SFT_CFG["grad_accum"],
        learning_rate=SFT_CFG["learning_rate"],
        warmup_ratio=SFT_CFG["warmup_ratio"],
        weight_decay=SFT_CFG["weight_decay"],
        max_length=SFT_CFG["max_seq_length"],
        bf16=True,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=20,
        save_strategy="no",
        seed=SFT_CFG["seed"],
        report_to=[],
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=sft_train,
        eval_dataset=sft_val,
        peft_config=peft_cfg,
        processing_class=tokenizer,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    trainer.save_model(str(SFT_OUT))
    tokenizer.save_pretrained(SFT_OUT)

    final = trainer.evaluate()
    summary = {
        "stage": "sft",
        "run": SFT_RUN,
        "learning_rate": SFT_CFG["learning_rate"],
        "trainable_params": trainable,
        "total_params": total,
        "epochs": SFT_CFG["epochs"],
        "train_examples": len(sft_train),
        "val_examples": len(sft_val),
        "minutes": round(minutes, 1),
        "final_eval_loss": round(final["eval_loss"], 4),
        "final_eval_perplexity": round(math.exp(final["eval_loss"]), 2),
    }
    (SFT_OUT / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1)
    )
    (SFT_OUT / "summary.json").write_text(json.dumps(summary, indent=1))
    print("done:", summary)
    return summary


In [6]:
# Runs supervised fine-tuning. Needs a CUDA GPU; about 10-15 minutes on the
# RTX 4090 used here.
summary_sft = run_sft()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/6771 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6771 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.615457,1.605443,1.627362,521808.000000,0.630701
200,1.561854,1.554676,1.565546,1045638.000000,0.638750
300,1.540232,1.532120,1.528089,1562484.000000,0.642821
400,1.514422,1.520714,1.498449,2083092.000000,0.644700
500,1.467840,1.511085,1.492090,2599214.000000,0.646390
600,1.477244,1.503980,1.471914,3119204.000000,0.647497
700,1.466934,1.499493,1.473476,3640152.000000,0.648091
800,1.445155,1.496419,1.448652,4159128.000000,0.648510
848,1.461716,1.495489,1.463951,4405448.000000,0.648832


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.461716,1.495489,848,1.463951,4405448.000000,0.648832


done: {'stage': 'sft', 'run': 'ma2-360m-sft', 'learning_rate': 0.0002, 'trainable_params': 8683520, 'total_params': 370504640, 'epochs': 2, 'train_examples': 6771, 'val_examples': 750, 'minutes': 17.4, 'final_eval_loss': 1.4955, 'final_eval_perplexity': 4.46}


## 3.5 Sanity check: sample generations

A handful of prompts are run through both the merged Mini-Assignment 1 model (the starting point of this stage) and the supervised fine-tuned model. Side by side, the supervised model should:

1. Recognise the request as an instruction to produce a complete structured posting, rather than continuing the input as free text.
2. Produce output with the expected headings (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`).
3. Stay on topic with respect to the recruiter query.

This is qualitative. The full multi-model comparison (base, supervised fine-tuned, and the four DPO variants) lives in Section 6.


In [5]:
# Quick sanity check: a few prompts through MA1-only (merged) and MA1+SFT,
# greedy decoding for reproducibility.
from peft import PeftModel

SANITY_PROMPTS = [
    "We are looking for a Senior Data Scientist capable of helping junior members",
    "We need to reforce our team with a DevSecOps Team Lead",
    "Looking for a Functional Tester that can also work with business requirements",
]

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

def generate(model, tokenizer, prompt: str, max_new_tokens: int = 400) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True
    )

# Load MA1-merged (the starting point) and MA1+SFT.
ma1_tok   = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1       = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
).to("cuda")
if ma1_tok.pad_token_id is not None:
    ma1.config.pad_token_id = ma1_tok.pad_token_id
sft_base  = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
)
sft_model = PeftModel.from_pretrained(sft_base, str(SFT_OUT)).to("cuda")
if ma1_tok.pad_token_id is not None:
    sft_model.config.pad_token_id = ma1_tok.pad_token_id

for q in SANITY_PROMPTS:
    p = build_prompt(q)
    print("=" * 80)
    print("REQUEST:", q)
    print("=" * 80)
    print("\n--- merged MA1 (text completer, no SFT) ---")
    print(generate(ma1, ma1_tok, p))
    print("\n--- MA1 + SFT ---")
    print(generate(sft_model, ma1_tok, p))
    print()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

REQUEST: We are looking for a Senior Data Scientist capable of helping junior members

--- merged MA1 (text completer, no SFT) ---
The post is open to all candidates who have at least 2 years experience with data science and/or machine learning
Requirements:

- Experience working as an Assistant or Recruiting Specialist; 
- Knowledgeable about the basics on statistics (mean / median);  
  - Basic knowledge how algorithms work from scratch using Python code ;
 – Understanding basic concepts like linear regression , logistic regressions etc.;

 — Good understanding of SQL queries & relational databases(MySQL)


**Responsibilities:**

— Work closely together with other team members including developers, QA engineers, product managers, designers, business analysts, project management teams, HR specialists, IT security experts, legal advisors, compliance officers, finance department, marketing departments, sales representatives, customer support staffs, internal consultants, external vendor

# 4. Preference dataset construction

Direct Preference Optimization needs a dataset of (prompt, chosen response, rejected response) triples. Rather than annotate by hand or pull a generic dataset, this project builds its own through reinforcement learning from AI feedback: candidates are sampled from the supervised fine-tuned model and ranked by a separate local LLM.

The judge is the `atlm_rlaif_judge` agent on `agent_server`. The rubric, sampling parameters, and per-family `chat_template_kwargs` are registered server-side, in the same way as the Mini-Assignment 1 `atlm_teacher` agent. The notebook only sends the user message; nothing is inlined in the code cells. Nemotron Nano 4B is made the active resident on `agent_server` immediately before the judging step.

The result of this section, `data/processed/ma2/preferences.jsonl`, is the input to the DPO trainer in Section 5.


## 4.0 Helpers for agent_server

Shared utility used by Section 4.3 (RLAIF listwise judge) and Section 6.4 (pairwise eval judge). `switch_active_model` makes a given local chat model the resident on `agent_server` via the admin API (see `~/env/assets/agent_server/documents/active_model_switching_sdk.md` section 3).

The two judges (`atlm_rlaif_judge` and `atlm_eval_judge`) are `agent_server` agent presets. Their system prompts, temperature, max_tokens, and per-family `chat_template_kwargs` live in `agent_server`'s registry, following the pattern in `~/env/assets/agent_server/documents/how_to.md`. Install the two presets from [`documents/development/agent_server_setup/`](../../documents/development/agent_server_setup/) before running the cells below.


In [8]:
# Section 4.0 helpers - one utility, shared by all the cells that call
# agent_server.
#
# Why this is the only helper here: the judging system prompt, sampling
# parameters, and per-family chat_template_kwargs are not in this notebook -
# they live in agent_server agent presets (atlm_rlaif_judge, atlm_eval_judge),
# following the pattern documented in
# ~/env/assets/agent_server/documents/how_to.md. Each judge cell only needs
# to (a) make sure agent_server has the correct chat model resident, then
# (b) call the agent by name with just a user message.
#
# Per the SDK doc (~/env/assets/agent_server/documents/active_model_switching_sdk.md
# section 3), the switch endpoint POSTs to /admin/api/active-model and the
# restart takes 30-45 s; we poll /v1/models until the new model serves.
import json
import time
import urllib.request

AGENT_SERVER  = "http://localhost:7701"
ENDPOINT      = f"{AGENT_SERVER}/v1/chat/completions"
ADMIN_SWITCH  = f"{AGENT_SERVER}/admin/api/active-model"
MODELS_URL    = f"{AGENT_SERVER}/v1/models"
SWITCH_SETTLE_TIMEOUT = 120
SWITCH_MIN_WAIT_S     = 60   # client-side minimum hold after a non-noop switch


def _http_get_json(url, timeout=10):
    with urllib.request.urlopen(urllib.request.Request(url), timeout=timeout) as r:
        return json.load(r)


def _http_post_json(url, body, timeout=30):
    req = urllib.request.Request(
        url, data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)


def switch_active_model(model_id):
    """Make `model_id` the active chat model on agent_server. Idempotent:
    no-op if already active. On a cold switch this POSTs to the admin
    endpoint, polls /v1/models until the new model is serving, AND holds
    for at least SWITCH_MIN_WAIT_S seconds total before returning (the
    client-side minimum hold the agent_server operator requested - even
    when /v1/models flips to the new model quickly, downstream calls
    issued before the hold has elapsed can race the restart). Raises
    RuntimeError on timeout."""
    t0 = time.time()
    try:
        data = _http_get_json(MODELS_URL)["data"]
        active = next(
            (m for m in data if m.get("kind") == "chat" and m.get("active")),
            None,
        )
        if active and active["id"] == model_id:
            print(f"  [switch] {model_id} already active; no-op")
            return 0.0
    except Exception:
        pass

    print(f"  [switch] POST /admin/api/active-model model_id={model_id}", flush=True)
    resp = _http_post_json(ADMIN_SWITCH, {"model_id": model_id})

    # Either way, hold at least SWITCH_MIN_WAIT_S after the switch request.
    # Per the agent_server operator: clients must allow ~1 minute before
    # the next call. Even a server-reported noop after a real swap still
    # needs the hold.
    deadline = t0 + SWITCH_SETTLE_TIMEOUT
    polls = 0
    serving = False
    while time.time() < deadline:
        time.sleep(3)
        polls += 1
        try:
            data = _http_get_json(MODELS_URL, timeout=5)["data"]
            active = next(
                (m for m in data if m.get("kind") == "chat" and m.get("active")),
                None,
            )
            if active and active["id"] == model_id:
                serving = True
                break
        except Exception:
            # agent_server is briefly unreachable during its own restart
            continue
    if not serving:
        raise RuntimeError(
            f"timed out after {SWITCH_SETTLE_TIMEOUT}s waiting for {model_id}"
        )

    # Hold to satisfy the client-side minimum wait.
    elapsed = time.time() - t0
    if elapsed < SWITCH_MIN_WAIT_S:
        remaining = SWITCH_MIN_WAIT_S - elapsed
        print(f"  [switch] /v1/models shows {model_id} active after {elapsed:.1f}s; "
              f"holding {remaining:.1f}s more to satisfy the {SWITCH_MIN_WAIT_S}s "
              f"client minimum...", flush=True)
        time.sleep(remaining)
    el = time.time() - t0
    print(f"  [switch] {model_id} ready after {el:.1f}s ({polls} polls)", flush=True)
    return el


## 4.1 Preference prompts

The prompts used to build the preference dataset come from the supervised fine-tuning validation records, the 250 records held out at the 90/10 split in Section 3.2. The model never trained on these, so the candidates sampled in Section 4.2 reflect generalisation rather than memorisation, which is what the preference signal should shape.

One query is picked per validation record. The records carry three phrasings of the same posting, but a single phrasing per posting is enough: using all three would give the judge near-duplicate prompts and waste calls. That yields 250 distinct recruiter prompts in `data/processed/ma2/preference_prompts.jsonl`. A defensive check confirms zero overlap with the 20 evaluation prompts from Section 6.1, which is guaranteed by construction (the evaluation prompts are hand-written, not part of `converted.jsonl`) but still worth verifying.


In [9]:
# Section 4.1 - select preference prompts from the SFT validation split.
# CPU only. Reproducible with seed 42.
import json
import random
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA     = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
MA2_DIR      = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_FILE    = MA2_DIR / "eval_prompts.jsonl"
PREF_PROMPTS = MA2_DIR / "preference_prompts.jsonl"
SEED = 42

# Reproduce the Section 3.2 record-level split (seed 42, 90/10) so the val
# records here are exactly the records the SFT model never trained on.
records = [json.loads(l) for l in open(SFT_DATA, encoding="utf-8")]
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records = records[:n_val]
print(f"SFT val records (held out from training): {len(val_records):,}")

# One query per record, picked deterministically with a separate seed so
# the query pick does not correlate with the record shuffle.
rng = random.Random(SEED + 1)
selected = []
for r in val_records:
    q_idx = rng.randrange(len(r["queries"]))
    selected.append({
        "prompt_id": f"pref-{r['id'].replace(':', '-')}",
        "query": r["queries"][q_idx],
        "source_record_id": r["id"],
        "source": r.get("source", "unknown"),
    })
print(f"Selected preference prompts: {len(selected):,}")

# Defensive check: no overlap with the evaluation prompt set.
eval_queries = {json.loads(l)["query"] for l in open(EVAL_FILE, encoding="utf-8")}
overlap = sum(1 for p in selected if p["query"] in eval_queries)
print(f"Overlap with eval prompts (expected 0): {overlap}")

# Persist.
MA2_DIR.mkdir(parents=True, exist_ok=True)
with open(PREF_PROMPTS, "w", encoding="utf-8") as f:
    for p in selected:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Wrote {PREF_PROMPTS.relative_to(PROJECT_ROOT)}")


SFT val records (held out from training): 250
Selected preference prompts: 250
Overlap with eval prompts (expected 0): 0
Wrote data/processed/ma2/preference_prompts.jsonl


## 4.2 Sampling candidates from the SFT model

For each of the 250 preference prompts, four candidates are sampled from the supervised fine-tuned model. Sampling at temperature 0.9 with top-p 0.95 rather than greedy decoding: the candidates have to differ enough that a judge can rank them. Greedy would give four near-identical samples and no preference signal.

Setting `num_return_sequences=4` returns the four candidates in a single batched `generate` call, which is the cheap way to do this on a small model. The new-token budget is 500, comfortably long enough for a structured Markdown posting.

Output goes to `data/processed/ma2/sft_candidates.jsonl`, one line per (prompt, candidate). The cell is idempotent: prompts that already have four candidates on disk are skipped, so the run is resumable. After sampling, the GPU is freed (the SFT model is unloaded, the CUDA cache cleared) so the judging step in Section 4.3 can use the GPU for the Nemotron chat model that `agent_server` will swap in.

The cell needs a CUDA GPU. On the RTX 4090 used here, 250 prompts at four candidates each take roughly 30 to 45 minutes.


In [10]:
# Section 4.2 - define sample_candidates(): sample k completions per
# preference prompt from the SFT model. Function is GPU-bound and is
# called by the next cell.
import gc
import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MERGED_DIR      = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"
SFT_OUT         = PROJECT_ROOT / "outputs" / "ma2-360m-sft"
MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
SEED = 42

# Same template as Section 3.1, defined here so Section 4 can run after a
# kernel restart without re-running Section 3.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

SAMPLING_CFG = {
    "k": 4,
    "max_new_tokens": 500,
    "temperature": 0.9,
    "top_p": 0.95,
    "seed": SEED,
}

def _done_prompt_ids(path, k):
    if not path.exists():
        return set()
    counts = {}
    for line in open(path, encoding="utf-8"):
        pid = json.loads(line)["prompt_id"]
        counts[pid] = counts.get(pid, 0) + 1
    return {pid for pid, n in counts.items() if n >= k}

def sample_candidates():
    set_seed(SAMPLING_CFG["seed"])

    # Load merged MA1 base + the SFT LoRA adapter on top, in bf16, on cuda.
    tok = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base  = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, str(SFT_OUT)).to("cuda").eval()

    prompts = [json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8")]
    done = _done_prompt_ids(CANDIDATES_FILE, SAMPLING_CFG["k"])
    todo = [p for p in prompts if p["prompt_id"] not in done]
    print(f"prompts total {len(prompts)} | already done {len(done)} | todo {len(todo)}")

    f = open(CANDIDATES_FILE, "a", encoding="utf-8")
    t0 = time.time()
    for i, p in enumerate(todo, 1):
        text = build_prompt(p["query"])
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=SAMPLING_CFG["max_new_tokens"],
                do_sample=True,
                temperature=SAMPLING_CFG["temperature"],
                top_p=SAMPLING_CFG["top_p"],
                num_return_sequences=SAMPLING_CFG["k"],
                pad_token_id=tok.eos_token_id,
            )
        in_len = enc["input_ids"].shape[1]
        for j, o in enumerate(out, 1):
            txt = tok.decode(o[in_len:], skip_special_tokens=True)
            f.write(json.dumps({"prompt_id": p["prompt_id"],
                                "candidate_idx": j,
                                "text": txt},
                               ensure_ascii=False) + "\n")
        f.flush()
        if i % 10 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  {i}/{len(todo)} prompts | {el/60:.1f} min | "
                  f"{i/max(el,1e-6):.2f} prompts/s")
    f.close()

    # Free the GPU so agent_server can use it for Gemma-4 in Section 4.3.
    del model, base, tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f"done in {(time.time()-t0)/60:.1f} min, GPU freed")


In [11]:
# Sample candidates from the SFT model. GPU-bound; roughly 30-45 min
# on the 4090. Idempotent: re-running picks up where it left off.
sample_candidates()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

prompts total 250 | already done 0 | todo 250
  10/250 prompts | 2.1 min | 0.08 prompts/s
  20/250 prompts | 4.2 min | 0.08 prompts/s
  30/250 prompts | 6.5 min | 0.08 prompts/s
  40/250 prompts | 8.7 min | 0.08 prompts/s
  50/250 prompts | 10.8 min | 0.08 prompts/s
  60/250 prompts | 13.0 min | 0.08 prompts/s
  70/250 prompts | 15.2 min | 0.08 prompts/s
  80/250 prompts | 17.4 min | 0.08 prompts/s
  90/250 prompts | 19.7 min | 0.08 prompts/s
  100/250 prompts | 21.9 min | 0.08 prompts/s
  110/250 prompts | 24.2 min | 0.08 prompts/s
  120/250 prompts | 26.4 min | 0.08 prompts/s
  130/250 prompts | 28.4 min | 0.08 prompts/s
  140/250 prompts | 30.7 min | 0.08 prompts/s
  150/250 prompts | 33.0 min | 0.08 prompts/s
  160/250 prompts | 35.0 min | 0.08 prompts/s
  170/250 prompts | 37.2 min | 0.08 prompts/s
  180/250 prompts | 39.5 min | 0.08 prompts/s
  190/250 prompts | 41.6 min | 0.08 prompts/s
  200/250 prompts | 43.7 min | 0.08 prompts/s
  210/250 prompts | 45.6 min | 0.08 prompts/s
 

## 4.3 Judging candidates with the `atlm_rlaif_judge` agent

For each prompt, the four candidates go to the `atlm_rlaif_judge` agent on `agent_server`. The agent's system prompt, sampling parameters and `chat_template_kwargs` are registered in `agent_server`'s preset registry (see [`documents/development/agent_server_setup/`](../../documents/development/agent_server_setup/) and `~/env/assets/agent_server/documents/how_to.md`), in the same way as the Mini-Assignment 1 `atlm_teacher`. The notebook sends `{model: "atlm_rlaif_judge", messages: [{role: user, content: ...}]}` and nothing else; `temperature=0.0`, `max_tokens=16384` and `chat_template_kwargs.enable_thinking=true` come from the preset.

The agent runs on whatever chat model is the active resident. `switch_active_model("nemotron")` is called first to make Nemotron Nano 4B that resident. Nemotron was picked from the calibration battery in [`documents/development/llm_models_performance.md`](../../documents/development/llm_models_performance.md): the only non-Gemma candidate that combined a 100 percent parse rate on the listwise task, real `<think>` reasoning, moderate disagreement with the Gemma baseline, and a tractable wall-time budget. Qwen3.5 blew the 16K-token cap, Granite emitted no reasoning, Ministral was bimodal and slow.

The judge is asked for a full ranking of all four candidates rather than only the best and worst. Each ranking yields six (higher, lower) pairs, so 250 prompts produce up to 1,500 preference triples at zero extra generation cost (the four candidates per prompt are sampled once and reused). The earlier best-worst variant of the rubric, used in Configurations B (v4_AC) and C (v5_AC), produced one pair per prompt; switching to the full ranking was a deliberate signal-volume increase tested in v6_AC. The judge ends its response with the single parseable line `RANKING: <best> > <2nd> > <3rd> > <worst>`.

The rubric, exactly as registered server-side:

```text
You are an expert recruiter and a careful writing reviewer.

You will be shown a recruiter request and four candidate job postings, numbered 1 to 4. Your task is to rank all four candidates from best to worst.

Apply the following criteria, in order of importance:

1. Faithfulness to the request: the posting must reflect the role, technology stack, seniority and any constraints expressed in the request.
2. Structure and completeness: the posting must include the four required sections '## Summary', '## Required Skills', '## Responsibilities' and '## Requirements', each with substantive, non-empty content.
3. Professional and inclusive language: clear, neutral, free of biased or discriminatory phrasing.
4. Absence of repetition, rambling, or truncation: each section must read cleanly without filler or mid-sentence stops.

You may briefly note the strengths and weaknesses of each candidate before deciding.

End your response with the single line giving the full ranking from best to worst:

RANKING: <best> > <2nd> > <3rd> > <worst>

where each placeholder is the candidate number (1, 2, 3 or 4). Every candidate must appear exactly once. Do not write anything after that line.
```

Output is `data/processed/ma2/judge_ranks.jsonl`, one line per ranked prompt, carrying the full ranking and the raw judge response (kept for traceability, including the `<think>` block). The step is idempotent: prompts already in the file are skipped. The judge call uses four worker threads.

The cell needs `agent_server` running on `http://localhost:7701` with `atlm_rlaif_judge` installed.


In [12]:
# Section 4.3 - define judge_all(): for each preference prompt, get a
# full listwise ranking of the four candidates from the atlm_rlaif_judge
# agent on agent_server. The agent supplies the system prompt, sampling,
# and chat_template_kwargs (per how_to.md Option A1).
import json
import queue
import re
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
JUDGE_RANKS     = MA2_DIR / "judge_ranks.jsonl"

JUDGE_MODEL   = "nemotron"
JUDGE_AGENT   = "atlm_rlaif_judge"
TIMEOUT       = 600
JUDGE_WORKERS = 4


def _build_judge_user(query, candidates):
    parts = [f"RECRUITER REQUEST:\n{query}\n"]
    for i, c in enumerate(candidates, 1):
        parts.append(f"\n---\n\nCANDIDATE {i}:\n{c}\n")
    return "".join(parts)


def _call_judge(query, candidates):
    """Call the atlm_rlaif_judge agent. Just {model, messages} - the
    agent preset supplies the rest. Per how_to.md Option A1."""
    payload = {
        "model": JUDGE_AGENT,
        "messages": [
            {"role": "user", "content": _build_judge_user(query, candidates)},
        ],
    }
    req = urllib.request.Request(
        ENDPOINT, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        resp = json.load(r)
    return resp["choices"][0]["message"]["content"]


# Full ranking line: "RANKING: A > B > C > D"
_RANKING_RE = re.compile(
    r"RANKING:\s*(\d)\s*>\s*(\d)\s*>\s*(\d)\s*>\s*(\d)",
    re.I,
)


def _parse_ranking(text):
    """Parse 'RANKING: A > B > C > D' into [A, B, C, D] (best..worst).
    Returns None if the line is missing, has the wrong arity, or contains
    a duplicate or out-of-range id."""
    m = _RANKING_RE.search(text or "")
    if not m:
        return None
    ranking = [int(m.group(i)) for i in range(1, 5)]
    if sorted(ranking) != [1, 2, 3, 4]:
        return None
    return ranking


def _done_judge_prompt_ids(path):
    if not path.exists():
        return set()
    return {json.loads(l)["prompt_id"] for l in open(path, encoding="utf-8")}


def judge_all():
    # Make sure agent_server has the right chat model resident for the
    # atlm_rlaif_judge agent. Idempotent: no-op if already active.
    switch_active_model(JUDGE_MODEL)

    # Build prompt_id -> {candidate_idx: text} from sft_candidates.jsonl.
    by_prompt = {}
    for l in open(CANDIDATES_FILE, encoding="utf-8"):
        r = json.loads(l)
        by_prompt.setdefault(r["prompt_id"], {})[r["candidate_idx"]] = r["text"]

    prompts = [json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8")]
    done = _done_judge_prompt_ids(JUDGE_RANKS)
    todo = [p for p in prompts
            if p["prompt_id"] not in done
            and len(by_prompt.get(p["prompt_id"], {})) == 4]
    print(f"prompts with 4 candidates: {sum(1 for v in by_prompt.values() if len(v)==4)} "
          f"| already judged: {len(done)} | todo: {len(todo)}")
    if not todo:
        print("nothing to do.")
        return

    q = queue.Queue()
    for p in todo:
        q.put(p)
    f = open(JUDGE_RANKS, "a", encoding="utf-8")
    try:
        lock = threading.Lock()
        stats = {"ok": 0, "parse_fail": 0, "err": 0}
        t0 = time.time()

        def worker():
        while True:
            try:
                p = q.get_nowait()
            except queue.Empty:
                return
            cands = by_prompt[p["prompt_id"]]
            cand_list = [cands[i] for i in (1, 2, 3, 4)]
            try:
                raw = _call_judge(p["query"], cand_list)
            except Exception:
                with lock:
                    stats["err"] += 1
                continue
            parsed = _parse_ranking(raw)
            with lock:
                if parsed is None:
                    stats["parse_fail"] += 1
                else:
                    f.write(json.dumps({
                        "prompt_id": p["prompt_id"],
                        "ranking":   parsed,   # [best, 2nd, 3rd, worst]
                        "raw":       raw,
                    }, ensure_ascii=False) + "\n")
                    f.flush()
                    stats["ok"] += 1
                n_seen = stats["ok"] + stats["parse_fail"] + stats["err"]
                if n_seen % 20 == 0:
                    el = time.time() - t0
                    print(f"  ok={stats['ok']} parse_fail={stats['parse_fail']} "
                          f"err={stats['err']} | {el/60:.1f} min")

        threads = [threading.Thread(target=worker, daemon=True)
               for _ in range(JUDGE_WORKERS)]
        for t in threads: t.start()
        for t in threads: t.join()
    finally:
        f.close()
    el = time.time() - t0
    print(f"done in {el/60:.1f} min | ok={stats['ok']} "
          f"parse_fail={stats['parse_fail']} err={stats['err']}")


In [13]:
# Judge all prompts. Requires agent_server up on http://localhost:7701.
# Idempotent: re-running picks up where it left off.
# Expected ~1-2 hours for 250 prompts at concurrency 4.
judge_all()


  [switch] POST /admin/api/active-model model_id=nemotron
  [switch] /v1/models shows nemotron active after 36.1s; holding 23.9s more to satisfy the 60s client minimum...
  [switch] nemotron ready after 60.0s (12 polls)
prompts with 4 candidates: 250 | already judged: 0 | todo: 250
  ok=20 parse_fail=0 err=0 | 1.1 min
  ok=40 parse_fail=0 err=0 | 1.9 min
  ok=60 parse_fail=0 err=0 | 2.8 min
  ok=80 parse_fail=0 err=0 | 3.7 min
  ok=100 parse_fail=0 err=0 | 4.9 min
  ok=119 parse_fail=1 err=0 | 6.8 min
  ok=136 parse_fail=4 err=0 | 9.3 min
  ok=155 parse_fail=5 err=0 | 10.5 min
  ok=175 parse_fail=5 err=0 | 12.2 min
  ok=193 parse_fail=7 err=0 | 13.5 min
  ok=213 parse_fail=7 err=0 | 14.6 min
  ok=233 parse_fail=7 err=0 | 15.8 min
done in 17.7 min | ok=241 parse_fail=9 err=0


## 4.4 Final preference dataset

The DPO trainer in Section 5 expects `(prompt, chosen, rejected)` triples. This cell joins the three intermediate files (the preference prompts, the SFT candidates, and the judge ranks) into that shape, expanded by the full-ranking format introduced in Section 4.3.

Each full ranking yields six ordered (higher, lower) pairs, so 250 prompts produce up to 1,500 triples. The actual count depends on how many rankings parsed cleanly: the v6_AC run parsed 241 of 250 prompts and produced 1,446 triples. Per triple:

- `prompt` is the formatted inference prompt for the recruiter query, ending with the template's `### Posting` marker, identical to the string the SFT model continued from when sampling.
- `chosen` is the higher-ranked candidate's text for that pair.
- `rejected` is the lower-ranked candidate's text.

Output goes to `data/processed/ma2/preferences.jsonl`, the input to Section 5's `DPOTrainer`. The 6.5x expansion versus the best-worst variant used in Configurations B and C (222 triples) was a deliberate test of whether preference-data volume was the bottleneck for DPO improvement; Section 5.6 and Section 7.3 report the answer.


In [14]:
# Section 4.4 - assemble the (prompt, chosen, rejected) preference
# dataset for DPO. From each full ranking [a, b, c, d] (best..worst)
# we extract all 6 (higher > lower) pairs, giving 6 triples per prompt.
# With 250 prompts that is 1,500 triples (vs 250 when using only best
# and worst). CPU only.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
JUDGE_RANKS     = MA2_DIR / "judge_ranks.jsonl"
PREFERENCES     = MA2_DIR / "preferences.jsonl"


PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)


def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )


def all_pairs(ranking):
    """For ranking [a, b, c, d] (best to worst) yield every (higher, lower)
    pair. Six pairs per 4-element ranking."""
    for i in range(len(ranking)):
        for j in range(i + 1, len(ranking)):
            yield ranking[i], ranking[j]


prompts = {p["prompt_id"]: p for p in
           (json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8"))}
cands = {}
for l in open(CANDIDATES_FILE, encoding="utf-8"):
    r = json.loads(l)
    cands.setdefault(r["prompt_id"], {})[r["candidate_idx"]] = r["text"]
ranks = [json.loads(l) for l in open(JUDGE_RANKS, encoding="utf-8")]

n_in = len(ranks)
n_out = 0
with open(PREFERENCES, "w", encoding="utf-8") as f:
    for r in ranks:
        pid = r["prompt_id"]
        if pid not in prompts or pid not in cands or len(cands[pid]) != 4:
            continue
        ranking = r["ranking"]
        prompt_text = build_prompt(prompts[pid]["query"])
        for chosen_id, rejected_id in all_pairs(ranking):
            f.write(json.dumps({
                "prompt":   prompt_text,
                "chosen":   cands[pid][chosen_id],
                "rejected": cands[pid][rejected_id],
            }, ensure_ascii=False) + "\n")
            n_out += 1
print(f"Wrote {n_out:,} preference triples from {n_in:,} ranked prompts "
      f"({n_out/max(n_in,1):.1f} pairs/prompt average) -> "
      f"{PREFERENCES.relative_to(PROJECT_ROOT)}")


Wrote 1,446 preference triples from 241 ranked prompts (6.0 pairs/prompt average) -> data/processed/ma2/preferences.jsonl


# 5. DPO training

Direct Preference Optimization aligns the supervised fine-tuned model so that, on the same prompt, the policy puts higher log-probability on the chosen response than on the rejected one. Compared with PPO it needs no separate reward model and no rollout loop, so it fits a single-GPU budget. We use HuggingFace TRL's `DPOTrainer`.

The starting point for this stage is the supervised fine-tuned model from Section 3. Following the same merge-between-stages pattern used at the Mini-Assignment 1 to SFT boundary, the SFT adapter is first merged into the base to produce a single consolidated checkpoint at `outputs/ma2-360m-sft-merged/`. A fresh LoRA is then trained on top of that for DPO.

Four DPO runs launch back to back from a single training cell (`ma2s54run`) at four beta values: 0.05, 0.10, 0.20 and 0.30. All other hyperparameters are held constant. Beta is the KL coefficient: lower beta lets the policy diverge further from the supervised fine-tuned reference (more aggressive alignment), higher beta keeps the policy closer to the reference (more conservative). Section 6 evaluates the resulting four DPO models alongside the base and the supervised fine-tuned model; Section 7.2 uses the four-point spread to locate the alignment-quality peak.


## 5.1 Merge the SFT adapter into the base

The SFT model (the merged Mini-Assignment 1 base plus the Section 3 LoRA adapter) is consolidated into a single self-contained checkpoint by calling `merge_and_unload`. This mirrors the Mini-Assignment 1 to SFT boundary and keeps the DPO stage simple: a fresh LoRA on a plain base, with no stacked-adapter bookkeeping.

CPU operation, idempotent, writes `outputs/ma2-360m-sft-merged/`. The result is the policy and, implicitly, the reference for the DPO stage below.


In [15]:
# Section 5.1 - merge the SFT LoRA into the merged MA1 base. CPU only.
import json
import gc
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MERGED_DIR     = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"
SFT_OUT        = PROJECT_ROOT / "outputs" / "ma2-360m-sft"
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"

t0 = time.time()
print(f"Base (merged MA1): {MERGED_DIR}")
print(f"SFT adapter      : {SFT_OUT}")
print(f"Output (SFT merged): {SFT_MERGED_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
base  = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, str(SFT_OUT), torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

SFT_MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_MERGED_DIR)
tokenizer.save_pretrained(SFT_MERGED_DIR)

(SFT_MERGED_DIR / "README.md").write_text(
    "# MA1+SFT merged into a single checkpoint\n\n"
    "This folder is the Mini-Assignment 1 LoRA (already merged in `outputs/mp1-360m/merged/`) "
    "plus the supervised fine-tuning LoRA from Section 3, both baked into the base weights, "
    "in bf16. Inference here is equivalent to loading the merged MA1 base and the SFT "
    "adapter together, with no PEFT plumbing.\n\n"
    "Provenance:\n"
    "- Base (merged MA1): `outputs/mp1-360m/merged/`\n"
    "- SFT adapter      : `outputs/ma2-360m-sft/`\n"
    "- Built by: notebook Section 5.1\n\n"
    "Purpose: starting point for the DPO stage in Section 5.\n"
)

del model, base, tokenizer
gc.collect()
print(f"Merged in {time.time() - t0:.1f}s.")


Base (merged MA1): /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged
SFT adapter      : /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft
Output (SFT merged): /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft-merged


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged in 4.0s.


## 5.2 Preference dataset

The `preferences.jsonl` file produced in Section 4.4 holds the (prompt, chosen, rejected) triples. The cell loads them and splits 90/10 at the triple level into a DPO training set and an evaluation set, with seed 42. The evaluation set is what the trainer uses to report the held-out preference metric (reward accuracy, the fraction of pairs where the policy log-probability margin agrees with the judge's choice), which feeds Section 5.6.


In [16]:
# Section 5.2 - load preferences.jsonl and split 90/10 for DPO training.
import json
import random
from pathlib import Path

from datasets import Dataset

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PREFERENCES = PROJECT_ROOT / "data" / "processed" / "ma2" / "preferences.jsonl"
SEED = 42

triples = [json.loads(l) for l in open(PREFERENCES, encoding="utf-8")]
print(f"Loaded {len(triples):,} preference triples")

random.Random(SEED).shuffle(triples)
n_val = max(1, len(triples) // 10)
val_triples   = triples[:n_val]
train_triples = triples[n_val:]
print(f"Split: {len(train_triples):,} train / {len(val_triples):,} eval")

dpo_train = Dataset.from_list(train_triples)
dpo_eval  = Dataset.from_list(val_triples)

# Quick check of the columns DPOTrainer expects.
print(f"Columns: {dpo_train.column_names}")


Loaded 1,446 preference triples
Split: 1,302 train / 144 eval
Columns: ['prompt', 'chosen', 'rejected']


## 5.3 Training configuration

The configuration mirrors Mini-Assignment 1 and the SFT stage where applicable. The DPO-specific values were reached through iteration documented in Section 7.2.

- Beta (KL coefficient). Four values, 0.05, 0.10, 0.20 and 0.30. The TRL and DPO-paper default is 0.10; the surrounding values explore both more aggressive (0.05) and more conservative (0.20, 0.30) policy divergence from the reference. All other hyperparameters held constant across the four runs.
- Learning rate 5e-5, an order of magnitude higher than the textbook DPO learning rate of 5e-6. LoRA-DPO updates a small subspace of the full parameter space and needs a proportionally higher learning rate to move the reward function. At 5e-6, reward margins were near zero across betas; at 1e-4 (tested in a v6 iteration), the policy collapsed into fluency degradation despite a higher training-side margin. 5e-5 is the sweet spot; Section 7.2 carries the full LR story.
- Five epochs over the 1,302 training triples, at effective batch 8 (micro-batch 2, gradient accumulation 4). Effective batch is halved from SFT's 16 because DPO forwards through both chosen and rejected at every step, raising memory pressure.
- LoRA shape r=16, alpha=32, dropout=0.05 across attention and feed-forward projections, identical to Mini-Assignment 1 and the SFT stage so the alignment delta is directly comparable.
- Maximum sequence length 1024, identical to SFT, with truncation when prompt plus response exceeds it.
- bf16, seed 42, no mid-run checkpoints.

The reference policy is left implicit. With a PEFT-wrapped policy passed to `DPOTrainer`, TRL computes reference log-probabilities by temporarily disabling the active adapter rather than loading a second model copy; this saves a model's worth of VRAM.


In [17]:
# Section 5.3 - DPO training configuration shared across both beta values.
DPO_BETAS = [0.05, 0.1, 0.2, 0.3]

DPO_CFG = {
    "epochs": 5,
    "per_device_batch": 2,
    "grad_accum": 4,                   # effective batch = 8
    "learning_rate": 5e-5,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "seed": SEED,
}
print("DPO config (shared across beta values):")
for k, v in DPO_CFG.items():
    print(f"  {k}: {v}")
print(f"beta values to run: {DPO_BETAS}")


DPO config (shared across beta values):
  epochs: 5
  per_device_batch: 2
  grad_accum: 4
  learning_rate: 5e-05
  warmup_ratio: 0.03
  weight_decay: 0.01
  max_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  seed: 42
beta values to run: [0.1, 0.3]


## 5.4 Running DPO

`run_dpo` loads the SFT-merged base (from Section 5.1) as the policy, wraps it in a fresh LoRA configured for DPO, and runs TRL's `DPOTrainer` for a given beta. The training cell `ma2s54run` calls `run_dpo` back to back for the four beta values. The cell is idempotent on `summary.json` existence: a run whose output directory already contains `summary.json` is skipped, so the cell can be re-executed safely (for example to add a new beta to an existing sweep without retraining the others, which is exactly how b005 was added late in v6_AC).

Each run writes its adapter, log history and a summary file to a tagged output directory: `outputs/ma2-360m-dpo-b005/`, `outputs/ma2-360m-dpo-b01/`, `outputs/ma2-360m-dpo-b02/`, `outputs/ma2-360m-dpo-b03/`.

The trainer's `log_history` captures the DPO-specific telemetry (reward margins, accuracies, KL to reference) per logging step, which goes into Section 5.6 and Section 7.2.

Each call needs a CUDA GPU. Expected wall-clock is roughly 30 minutes per beta on the RTX 4090 used here, so the four back-to-back calls take roughly two hours from a cold cache.


In [36]:
# Section 5.4 - defines run_dpo(): one DPO run for a given beta value,
# producing a fresh LoRA adapter at outputs/<run_tag>/.
import json
import math
import time
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# Same path resolution as the cells above.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"

def run_dpo(beta: float, run_tag: str):
    set_seed(DPO_CFG["seed"])
    out_dir = PROJECT_ROOT / "outputs" / run_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load the SFT-merged checkpoint as the DPO policy starting point.
    tokenizer = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED_DIR, torch_dtype=torch.bfloat16
    )

    dpo_lora = LoraConfig(
        r=DPO_CFG["lora_r"],
        lora_alpha=DPO_CFG["lora_alpha"],
        lora_dropout=DPO_CFG["lora_dropout"],
        target_modules=DPO_CFG["lora_target_modules"],
        task_type="CAUSAL_LM",
    )

    args = DPOConfig(
        output_dir=str(out_dir),
        num_train_epochs=DPO_CFG["epochs"],
        per_device_train_batch_size=DPO_CFG["per_device_batch"],
        per_device_eval_batch_size=DPO_CFG["per_device_batch"],
        gradient_accumulation_steps=DPO_CFG["grad_accum"],
        learning_rate=DPO_CFG["learning_rate"],
        warmup_ratio=DPO_CFG["warmup_ratio"],
        weight_decay=DPO_CFG["weight_decay"],
        max_length=DPO_CFG["max_length"],
        bf16=True,
        eval_strategy="steps",
        eval_steps=10,
        logging_steps=5,
        save_strategy="no",
        seed=DPO_CFG["seed"],
        report_to=[],
        beta=beta,
        loss_type="sigmoid",
        remove_unused_columns=False,    # keep prompt/chosen/rejected columns
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,                 # implicit via PEFT
        args=args,
        train_dataset=dpo_train,
        eval_dataset=dpo_eval,
        peft_config=dpo_lora,
        processing_class=tokenizer,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(out_dir)

    final = trainer.evaluate()
    summary = {
        "stage": "dpo",
        "run": run_tag,
        "beta": beta,
        "learning_rate": DPO_CFG["learning_rate"],
        "trainable_params": trainable,
        "total_params": total,
        "epochs": DPO_CFG["epochs"],
        "train_examples": len(dpo_train),
        "val_examples": len(dpo_eval),
        "minutes": round(minutes, 1),
        "final_eval_loss": round(final.get("eval_loss", 0.0), 4),
    }
    # Capture any DPO-specific eval metrics (rewards/accuracies/margins/kl).
    for k, v in final.items():
        if isinstance(v, (int, float)) and k not in summary:
            summary[k] = round(v, 4)

    (out_dir / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1)
    )
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=1))

    # Free GPU before the next beta run.
    del trainer, model, tokenizer, dpo_lora
    gc.collect()
    torch.cuda.empty_cache()

    print("done:", summary)
    return summary


In [37]:
# Train b005/b01/b02/b03 back to back. Each call writes a self-contained
# outputs/ma2-360m-dpo-{tag}/ directory. Idempotent: skips if summary.json
# already exists (lets us add b005 without retraining the others).
from pathlib import Path as _P

_PR = _P.cwd()
while not (_PR / 'data').exists() and _PR != _PR.parent:
    _PR = _PR.parent

def _maybe_run(beta, tag):
    out = _PR / 'outputs' / tag / 'summary.json'
    if out.exists():
        print(f'skip {tag}: {out} exists')
        return None
    return run_dpo(beta, tag)

summary_dpo_b005 = _maybe_run(0.05, 'ma2-360m-dpo-b005')
summary_dpo_b01  = _maybe_run(0.10, 'ma2-360m-dpo-b01')
summary_dpo_b02  = _maybe_run(0.20, 'ma2-360m-dpo-b02')
summary_dpo_b03  = _maybe_run(0.30, 'ma2-360m-dpo-b03')


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1302 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1302 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/144 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/144 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
10,0.687652,0.693480,1.651532,50777.000000,1.440181,1.477727,0.676905,-0.000984,-0.001061,0.520833,0.000077,-309.811245,-310.316026
20,0.698263,0.690863,1.650229,101627.000000,1.436734,1.475093,0.676687,0.005694,0.000441,0.562500,0.005253,-309.677693,-310.286002
30,0.692864,0.690037,1.648718,153287.000000,1.433956,1.470334,0.676703,0.007612,0.000681,0.569444,0.006931,-309.639320,-310.281193
40,0.688024,0.692363,1.649684,203949.000000,1.445418,1.483185,0.676950,0.002518,-0.000060,0.520833,0.002579,-309.741199,-310.296019
50,0.677408,0.690155,1.652216,254519.000000,1.483783,1.520755,0.676766,-0.006777,-0.013973,0.534722,0.007196,-309.927101,-310.574277
60,0.688911,0.682498,1.653454,307124.000000,1.515837,1.553550,0.677373,-0.019869,-0.043164,0.590278,0.023295,-310.188956,-311.158099
70,0.690426,0.682762,1.652016,358049.000000,1.532040,1.569846,0.677130,-0.034749,-0.058558,0.618056,0.023809,-310.486547,-311.465968
80,0.663457,0.678794,1.652374,410071.000000,1.563481,1.601474,0.677093,-0.055533,-0.088414,0.583333,0.032881,-310.902226,-312.063093
90,0.669463,0.677379,1.652183,460691.000000,1.583832,1.622362,0.676885,-0.085799,-0.122889,0.590278,0.037089,-311.507556,-312.752585
100,0.675949,0.675340,1.652417,513430.000000,1.600517,1.639651,0.674977,-0.123255,-0.166408,0.583333,0.043154,-312.256660,-313.622982


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.186218,0.654938,815,1.047326,4181080.000000,-2.666782,-2.574212,0.579455,-8.822625,-9.991270,0.701389,1.168645,-486.244067,-510.120209


done: {'stage': 'dpo', 'run': 'ma2-360m-dpo-b005', 'beta': 0.05, 'learning_rate': 5e-05, 'trainable_params': 8683520, 'total_params': 370504640, 'epochs': 5, 'train_examples': 1302, 'val_examples': 144, 'minutes': 29.7, 'final_eval_loss': 0.6549, 'eval_loss': 0.6549, 'eval_entropy': 1.0473, 'eval_num_tokens': 4181080.0, 'eval_logits/chosen': -2.6668, 'eval_logits/rejected': -2.5742, 'eval_mean_token_accuracy': 0.5795, 'eval_rewards/chosen': -8.8226, 'eval_rewards/rejected': -9.9913, 'eval_rewards/accuracies': 0.7014, 'eval_rewards/margins': 1.1686, 'eval_logps/chosen': -486.2441, 'eval_logps/rejected': -510.1202}
skip ma2-360m-dpo-b01: /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-dpo-b01/summary.json exists
skip ma2-360m-dpo-b02: /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-dpo-b02/summary.json exists
skip ma2-360m-dpo-b03: /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-dpo-b03/summary.json exists


## 5.5 Sanity check: sample generations

A small set of prompts is run through four model states side by side: the supervised fine-tuned model (the DPO starting point) and the three DPO-aligned models at beta = 0.10, 0.20 and 0.30. Greedy decoding with `repetition_penalty=1.3`, matching the inference kwargs used in Section 6.2. The fourth DPO variant at beta = 0.05 is reserved for the Section 6 quantitative comparison; the sanity check focuses on the in-range betas for a quick eyeball.

Expected qualitative differences: the DPO models should produce posting structure closer to the judge's preferred candidates (cleaner section headings, less rambling, fewer truncations), and the lower-beta model should diverge more visibly from SFT than the higher-beta one.

This is qualitative. The full quantitative comparison (perplexity, LLM-as-judge win-rate, behavioural statistics) on the 20 frozen prompts lives in Section 6.


In [6]:
# Section 5.5 - quick sanity check: SFT vs DPO-b01 vs DPO-b02 vs DPO-b03 on
# a few prompts, greedy decoding.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B02_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b02"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"

PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)


def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )


SANITY_PROMPTS = [
    "We are looking for a Senior Data Scientist capable of helping junior members",
    "We need to reforce our team with a DevSecOps Team Lead",
    "Looking for a Functional Tester that can also work with business requirements",
]


def generate(model, tokenizer, prompt: str, max_new_tokens: int = 400) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True,
    )


tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
sft_only = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
).to("cuda")

base_b01 = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
)
dpo_b01 = PeftModel.from_pretrained(base_b01, str(DPO_B01_DIR)).to("cuda")

base_b02 = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
)
dpo_b02 = PeftModel.from_pretrained(base_b02, str(DPO_B02_DIR)).to("cuda")

base_b03 = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
)
dpo_b03 = PeftModel.from_pretrained(base_b03, str(DPO_B03_DIR)).to("cuda")

for q in SANITY_PROMPTS:
    p = build_prompt(q)
    print("=" * 78)
    print(f"PROMPT: {q}")
    print("=" * 78)
    print("\n--- SFT only ---")
    print(generate(sft_only, tok, p))
    print("\n--- DPO beta=0.1 ---")
    print(generate(dpo_b01, tok, p))
    print("\n--- DPO beta=0.2 ---")
    print(generate(dpo_b02, tok, p))
    print("\n--- DPO beta=0.3 ---")
    print(generate(dpo_b03, tok, p))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PROMPT: We are looking for a Senior Data Scientist capable of helping junior members

--- SFT only ---
# Junior/Senior Machine Learning Engineer (Data Science)

## Summary
This role involves working closely with teams to develop and implement data science solutions within the financial services industry. The ideal candidate is eager to learn advanced machine learning techniques while contributing directly towards product development efforts across various projects. This position requires strong analytical skills combined with an entrepreneurial mindset focused on building scalable business models from scratch.

## Required Skills
- Deep understanding of statistical concepts such as regression analysis or time series forecasting methods like ARIMA modeling
- Proficiency in Python programming language including NumPy arrays and Pandas datasets handling
- Experience implementing deep neural networks using PyTorch framework
- Knowledge of modern ML algorithms used by large enterprises: Gra

## 5.6 Training results and hyperparameter sensitivity

Training-side metrics for the four DPO runs on the held-out preference validation set, read from the `summary.json` files in `outputs/ma2-360m-dpo-{b005,b01,b02,b03}/`:

| metric | beta = 0.05 | beta = 0.10 | beta = 0.20 | beta = 0.30 |
|---|---|---|---|---|
| eval_loss | 0.655 | 0.658 | 0.640 | 0.639 |
| eval reward accuracy | 0.701 | 0.715 | 0.694 | 0.694 |
| eval reward margin | 1.169 | 1.443 | 1.793 | 2.093 |
| eval entropy | 1.047 | 1.209 | 1.413 | 1.502 |
| eval mean token accuracy | 0.580 | 0.621 | 0.649 | 0.660 |
| training minutes | 29.7 | 29.5 | 29.1 | 29.1 |

The reward margin row scales linearly with beta by definition of the DPO loss (margin equals beta times the chosen-minus-rejected log-ratio), so the larger-margin-at-higher-beta pattern is a tautology, not a finding. The comparable cross-beta metric is the eval reward accuracy, which peaks at beta = 0.10 (0.715) and is slightly lower at both 0.05 (0.701) and the higher betas (0.694 each). On the training-distribution preference set, beta = 0.10 generalises best.

Entropy and mean token accuracy both decrease as beta drops, with beta = 0.05 producing the most confident, most divergent-from-SFT policy. That is the expected direction of the KL-coefficient trade-off; it is also the precondition for mode-collapse failure modes at inference, discussed in Section 7.3.

These training-side numbers measure how well the policy ranks chosen above rejected on training-distribution preferences. They do not, by themselves, identify the better-aligned model on the held-out 20-prompt deployment evaluation: that is Sections 6.3 to 6.8. The relationship between training-side ranking and deployment-side ranking is the topic of Section 7.

### 5.6.1 Hyperparameter sensitivity

The deliverable configuration (5 epochs, learning rate 5e-5, LoRA r=16, alpha=32, dropout=0.05) was reached through iteration. The learning rate was the deciding lever, with three concrete data points:

- 5e-6, the textbook DPO default for full fine-tuning: near-zero reward margins on this LoRA-DPO setup. The policy did not move.
- 5e-5, the deliverable: clean positive margins, monotonic improvement in eval reward accuracy across the in-range betas. Section 6.4 confirms the same direction on win-rate.
- 1e-4, an aborted v6 iteration documented in `documents/development/ma2_implementation_report.md` section 11.1: higher training-side reward margin than 5e-5, but visible fluency degradation on the held-out evaluation, including a hallucinated phrase "first-in, first-out" appearing across multiple unrelated prompts at beta = 0.1, with perplexity rising from 4.64 (Configuration C baseline) to 8.15.

Lifting the learning rate one order of magnitude over the textbook default is necessary for LoRA-DPO to move the reward function on this setup; lifting it two orders of magnitude is too far. The combination (beta = 0.10, learning rate = 5e-5) is the hyperparameter answer this work supports for LoRA-DPO on SmolLM2-360M.


# 6. Comparative evaluation

The final comparison contrasts the six model states the pipeline produced on the same fixed set of 20 prompts: the untrained base, the Mini-Assignment 1 model after supervised fine-tuning, and the four DPO-aligned models from Section 5 at beta = 0.05, 0.10, 0.20 and 0.30. Both automatic metrics (perplexity on a held-out set, LLM-as-judge win-rate with order-swap control) and qualitative side-by-side examples are reported.


## 6.1 Evaluation prompt set

The evaluation runs against the same fixed set of 20 recruiter queries on every model state. Fixing the set up front, before any further training, removes the temptation to cherry-pick favourable prompts after the fact and keeps the comparison honest.

The set is split into two ten-prompt sub-sets that test different things:

1. Held-out in-distribution. Ten queries that match the style and topic mix of the supervised fine-tuning data (`data/processed/converted.jsonl`) but are not part of it. These probe how well each model handles the kind of request it was trained on; they are the fair comparison for in-domain quality.
2. Fresh out-of-distribution. Ten hand-written queries that probe generalisation: unusual roles, niche tech stacks, atypical phrasings and soft-skill emphasis. These probe how well alignment carries over to prompts that look different from the training distribution, and where it may regress.

The two sub-sets are reported separately throughout Section 6 so the report can comment on each setting. The set is held out from the supervised fine-tuning data and from the Section 4 preference prompts, which makes it the only evaluation input the six model states share.


In [39]:
# Three-way evaluation prompt set: 20 queries, frozen here, used unchanged
# by every comparison in Section 6. Held out from SFT training data and from
# the preference-generation prompts in Section 4.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_PROMPTS = [
    # --- In-distribution: 10 recruiter queries in the style of converted.jsonl ---
    {"id": "ind-01", "subset": "in_distribution",
     "query": "We need a backend engineer to build and maintain Python services on AWS, with hands-on experience in Django and PostgreSQL."},
    {"id": "ind-02", "subset": "in_distribution",
     "query": "Looking for a React developer who can lead frontend architecture on a TypeScript codebase and mentor junior team members."},
    {"id": "ind-03", "subset": "in_distribution",
     "query": "We need an iOS engineer to ship a consumer-facing app in Swift, with experience in SwiftUI and Core Data."},
    {"id": "ind-04", "subset": "in_distribution",
     "query": "Looking for a machine learning engineer to put recommendation models into production, comfortable with PyTorch and Kubernetes."},
    {"id": "ind-05", "subset": "in_distribution",
     "query": "We need a data analyst to build dashboards and explore product data using SQL, dbt and Looker."},
    {"id": "ind-06", "subset": "in_distribution",
     "query": "Looking for a QA automation engineer with Selenium and Playwright experience, to own end-to-end testing of a web platform."},
    {"id": "ind-07", "subset": "in_distribution",
     "query": "We need a cloud engineer to design and operate AWS infrastructure using Terraform, EKS and modern observability tooling."},
    {"id": "ind-08", "subset": "in_distribution",
     "query": "Looking for a full-stack developer comfortable with Node.js, Next.js and PostgreSQL, to ship product features end to end."},
    {"id": "ind-09", "subset": "in_distribution",
     "query": "We need an embedded software engineer to write firmware in C and Rust for low-power IoT devices."},
    {"id": "ind-10", "subset": "in_distribution",
     "query": "Looking for a security engineer to lead application security, threat modelling and code review across multiple product teams."},

    # --- Out-of-distribution: 10 fresh hand-written queries probing generalisation ---
    {"id": "ood-01", "subset": "out_of_distribution",
     "query": "We are hiring a junior developer straight out of a coding bootcamp; the role is heavy on mentorship and pair programming, and the stack is whatever the team is using."},
    {"id": "ood-02", "subset": "out_of_distribution",
     "query": "Looking for a principal staff engineer who has scaled a backend from one to many regions and can act as the org-wide technical authority on distributed systems."},
    {"id": "ood-03", "subset": "out_of_distribution",
     "query": "We need a developer comfortable with Elixir and Phoenix for a real-time messaging product; OTP experience is essential."},
    {"id": "ood-04", "subset": "out_of_distribution",
     "query": "Looking for a quantitative developer to build low-latency trading systems in C++, with a strong background in numerical methods."},
    {"id": "ood-05", "subset": "out_of_distribution",
     "query": "We have a six-month contract for a freelance technical writer who can document a Rust SDK and produce API reference material."},
    {"id": "ood-06", "subset": "out_of_distribution",
     "query": "Looking for someone who is part data engineer and part analytics engineer, comfortable owning the pipeline from ingestion through dbt models to stakeholder dashboards."},
    {"id": "ood-07", "subset": "out_of_distribution",
     "query": "We need a developer who can explain complex topics to non-technical stakeholders, write clear documentation and run customer-facing workshops as part of the role."},
    {"id": "ood-08", "subset": "out_of_distribution",
     "query": "Hiring fully remote across European time zones with an asynchronous-first culture; we need a senior Go engineer to extend our core platform."},
    {"id": "ood-09", "subset": "out_of_distribution",
     "query": "Looking for a game engine programmer comfortable with Unreal Engine 5, gameplay systems and tooling in C++."},
    {"id": "ood-10", "subset": "out_of_distribution",
     "query": "We need a Rust developer to build WebAssembly modules that run client-side in the browser, with experience in performance-critical numerical code."},
]

# Persist as the single source of truth read by the rest of Section 6.
EVAL_DIR  = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE = EVAL_DIR / "eval_prompts.jsonl"
with open(EVAL_FILE, "w", encoding="utf-8") as f:
    for p in EVAL_PROMPTS:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

n_ind = sum(1 for p in EVAL_PROMPTS if p["subset"] == "in_distribution")
n_ood = sum(1 for p in EVAL_PROMPTS if p["subset"] == "out_of_distribution")
print(f"Wrote {len(EVAL_PROMPTS)} evaluation prompts to "
      f"{EVAL_FILE.relative_to(PROJECT_ROOT)}")
print(f"  in_distribution:     {n_ind}")
print(f"  out_of_distribution: {n_ood}")


Wrote 20 evaluation prompts to data/processed/ma2/eval_prompts.jsonl
  in_distribution:     10
  out_of_distribution: 10


## 6.2 Generation: producing outputs from each model

For each of the six model states (base SmolLM2-360M, the supervised fine-tuned model, and the four DPO-aligned models at beta = 0.05, 0.10, 0.20 and 0.30), greedy decoding is run on every one of the 20 evaluation prompts. Greedy rather than sampled decoding so the six-way comparison is reproducible and reflects each model's most-likely behaviour rather than a sample of its distribution.

The generation call carries `repetition_penalty=1.3`, matching the inference helper `src/generate_mp1.py` used in Mini-Assignment 1 for the same model family. This kwarg is necessary on small continued-pretrained models to suppress token-level repetition loops at the decoder. Section 7.3 documents what happens when this kwarg is omitted, which is what initially happened in v6_AC and what produced the catastrophic outputs that motivated the diagnostic pass.

The base model has never seen the Section 3.1 prompt template and is not an instruction follower; its generations are expected to read as raw text continuations rather than structured postings. That is the point: the comparison should make explicit what supervised fine-tuning and DPO added beyond pretraining.

Outputs go to `data/processed/ma2/eval_generations/<model>.jsonl`, one line per prompt with the prompt id, the subset, the original query, and the generation. Sections 6.4, 6.5 and 6.6 consume these files; the same outputs feed win-rate judging, qualitative display and behavioural statistics.


In [40]:
# Section 6.2 - generate from each of the five models on the 20 eval
# prompts. Greedy decoding with repetition_penalty=1.3, max_new_tokens=800.
# repetition_penalty was carried in MA1's generate_mp1.py for the same
# model family ("small continued-pretrained models loop easily") and was
# omitted from the MA2 evaluation function in v3-v6 by oversight. Restored
# here to suppress small-model decoder-loop attractors at inference time.
# GPU; ~12 min total.
import gc
import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR        = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_FILE      = MA2_DIR / "eval_prompts.jsonl"
GEN_DIR        = MA2_DIR / "eval_generations"
GEN_DIR.mkdir(parents=True, exist_ok=True)

SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B02_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b02"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"
DPO_B005_DIR   = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b005"
BASE_ID        = "HuggingFaceTB/SmolLM2-360M"


PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)


def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )


def load_model(tag: str):
    if tag == "base":
        tok = AutoTokenizer.from_pretrained(BASE_ID)
        m   = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16)
        return m.to("cuda").eval(), tok
    if tag == "sft":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        m   = AutoModelForCausalLM.from_pretrained(
            SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
        )
        return m.to("cuda").eval(), tok
    if tag in ("dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"):
        adapter_dir = {"dpo-b005": DPO_B005_DIR,
                       "dpo-b01": DPO_B01_DIR,
                       "dpo-b02": DPO_B02_DIR,
                       "dpo-b03": DPO_B03_DIR}[tag]
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        base = AutoModelForCausalLM.from_pretrained(
            SFT_MERGED_DIR, torch_dtype=torch.bfloat16,
        )
        model = PeftModel.from_pretrained(base, str(adapter_dir))
        return model.to("cuda").eval(), tok
    raise ValueError(f"unknown tag: {tag}")


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def gen_one(model, tok, prompt: str, max_new_tokens: int = 800) -> str:
    enc = tok(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)


prompts = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8")]
print(f"Evaluation prompts: {len(prompts)}")

TAGS = ["base", "sft", "dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"]
for tag in TAGS:
    out_path = GEN_DIR / f"{tag}.jsonl"
    if out_path.exists():
        print(f"=== skip {tag}: {out_path.name} exists ===", flush=True)
        continue
    print(f"\n=== generating from '{tag}' -> {out_path.name} ===", flush=True)
    t0 = time.time()
    model, tok = load_model(tag)
    with open(out_path, "w", encoding="utf-8") as f:
        for p in prompts:
            text = gen_one(model, tok, build_prompt(p["query"]))
            f.write(json.dumps({
                "prompt_id": p["id"],
                "subset":    p["subset"],
                "query":     p["query"],
                "generation": text,
            }, ensure_ascii=False) + "\n")
    del model, tok
    free_gpu()
    print(f"  done in {(time.time() - t0)/60:.1f} min")


Evaluation prompts: 20
=== skip base: base.jsonl exists ===
=== skip sft: sft.jsonl exists ===

=== generating from 'dpo-b005' -> dpo-b005.jsonl ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  done in 2.1 min
=== skip dpo-b01: dpo-b01.jsonl exists ===
=== skip dpo-b02: dpo-b02.jsonl exists ===
=== skip dpo-b03: dpo-b03.jsonl exists ===


## 6.3 Perplexity on the supervised fine-tuning validation set

Perplexity is the standard automatic metric for a language model. We compute it on the supervised fine-tuning validation set (the 250 held-out records, three queries each, 750 examples), formatted with the Section 3.1 template. This measures how well each of the six models fits the instruction-following distribution.

The base model is expected to have substantially higher perplexity than the other five, because it has never seen the prompt template and is not an instruction follower; its generations on the formatted prompt are out of distribution for it. The supervised fine-tuned model sets the floor on SFT-distribution perplexity, and the DPO models should land close to it (DPO is supposed to shift preferences, not destroy general fluency). Along the beta axis, higher beta keeps the policy closer to the SFT distribution and so should have lower perplexity; the lowest beta (0.05) should produce the largest divergence and the largest perplexity rise.

Numbers go to `data/processed/ma2/eval_perplexity.json`, read by Section 6.7.


In [41]:
# Section 6.3 - per-model perplexity on the SFT validation set. GPU.
import gc
import json
import math
import random
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA       = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
MA2_DIR        = PROJECT_ROOT / "data" / "processed" / "ma2"
OUT_FILE       = MA2_DIR / "eval_perplexity.json"

SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B02_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b02"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"
DPO_B005_DIR   = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b005"
BASE_ID        = "HuggingFaceTB/SmolLM2-360M"

SEED = 42
MAX_LEN = 1024


PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)


def format_example(query, jd):
    return f"{PROMPT_PREAMBLE}\n\n### Request\n{query}\n\n### Posting\n{jd}"


records = [json.loads(l) for l in open(SFT_DATA, encoding="utf-8")]
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records = records[:n_val]
val_texts = []
for r in val_records:
    for q in r["queries"]:
        val_texts.append(format_example(q, r["job_description"]))
print(f"SFT validation examples: {len(val_texts)}")


def _ensure_pad(tok):
    """Some tokenizers (e.g. raw SmolLM2-360M base) have no pad_token by
    default. perplexity() below uses padding=True which requires one."""
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model(tag: str):
    if tag == "base":
        tok = _ensure_pad(AutoTokenizer.from_pretrained(BASE_ID))
        m   = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=torch.bfloat16)
        return m.to("cuda").eval(), tok
    if tag == "sft":
        tok = _ensure_pad(AutoTokenizer.from_pretrained(SFT_MERGED_DIR))
        m   = AutoModelForCausalLM.from_pretrained(SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
        return m.to("cuda").eval(), tok
    if tag in ("dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"):
        adapter_dir = {"dpo-b005": DPO_B005_DIR,
        "dpo-b01": DPO_B01_DIR,
                       "dpo-b02": DPO_B02_DIR,
                       "dpo-b03": DPO_B03_DIR}[tag]
        tok = _ensure_pad(AutoTokenizer.from_pretrained(SFT_MERGED_DIR))
        base = AutoModelForCausalLM.from_pretrained(SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
        m   = PeftModel.from_pretrained(base, str(adapter_dir))
        return m.to("cuda").eval(), tok
    raise ValueError(f"unknown tag: {tag}")


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


@torch.inference_mode()
def perplexity(model, tok, texts, max_len=MAX_LEN, batch_size=4):
    tot_loss, tot_tok = 0.0, 0
    pad = tok.eos_token_id if tok.pad_token_id is None else tok.pad_token_id
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=max_len).to("cuda")
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100
        out = model(**enc, labels=labels)
        n = (labels != -100).sum().item() - labels.size(0)
        tot_loss += out.loss.item() * n
        tot_tok  += n
    return math.exp(tot_loss / tot_tok)


TAGS = ["base", "sft", "dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"]
results = {}
for tag in TAGS:
    model, tok = load_model(tag)
    pp = perplexity(model, tok, val_texts)
    results[tag] = round(pp, 2)
    print(f"  {tag:8s}  PPL = {pp:.2f}")
    del model, tok
    free_gpu()

summary = {
    "evaluation_set": "sft_val (250 records x 3 queries = 750 examples)",
    "max_length": MAX_LEN,
    "perplexity": results,
}
OUT_FILE.write_text(json.dumps(summary, indent=1))
print(f"\nWrote {OUT_FILE.relative_to(PROJECT_ROOT)}")
print(f"summary: {results}")


SFT validation examples: 750


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  base      PPL = 11.66


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  sft       PPL = 4.54


...

  dpo-b01   PPL = 6.01


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  dpo-b02   PPL = 5.03


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  dpo-b03   PPL = 4.80

Wrote data/processed/ma2/eval_perplexity.json
summary: {'base': 11.66, 'sft': 4.54, 'dpo-b005': 8.84, 'dpo-b01': 6.01, 'dpo-b02': 5.03, 'dpo-b03': 4.8}


## 6.4 LLM-as-judge win-rate with order-swap control

Pairwise win-rates are computed between every distinct pair of the six model states. Six models give 15 pairs (every combination of {base, SFT, DPO-b005, DPO-b01, DPO-b02, DPO-b03}). Times 20 evaluation prompts, times two orderings (A-then-B and B-then-A) for position-bias control, this comes to 600 judge calls.

The judge is the `atlm_eval_judge` agent on `agent_server`. Like the RLAIF judge in Section 4, the preset carries the rubric, `temperature=0.0`, `max_tokens=16384`, and `chat_template_kwargs.thinking=true` (Granite's family uses `thinking`, not `enable_thinking`, per SDK section 7b).

`switch_active_model("granite-3.3")` is called first to make Granite 3.3 2B the active resident. Granite was picked from the calibration battery in [`documents/development/llm_models_performance.md`](../../documents/development/llm_models_performance.md): 5 of 5 parse, 4 of 5 AB-BA consistent, 4 of 4 agreement with the Gemma baseline on cleanly-decidable pairs, caught the deliberately-broken `ood-03` collapse, and fast. Picking Granite (a different family than Nemotron, the RLAIF judge) keeps the eval judge independent of the preference judge, so the win-rate cannot reduce to a same-model self-preference loop. What the 10-call calibration probe did not surface, namely a structural position-bias discrimination ceiling on close pairs at this parameter budget, is the subject of Section 7.3.

Granite emits its reasoning as plain response text rather than wrapped in `<think>` tags; both the verdict and its justification land in the raw response stored in `winrate_calls.jsonl`.

The pairwise rubric, exactly as registered server-side:

```text
You are an expert recruiter and a careful writing reviewer.

You will be shown a recruiter request and two candidate job postings, labelled A and B. Your task is to decide which is better overall.

Apply the following criteria, in order of importance:

1. Faithfulness to the request: the posting must reflect the role, technology stack, seniority and any constraints expressed in the request.
2. Absence of repetition, rambling, or truncation: a posting that contains a repetition loop, padded lists of near-duplicate items, or mid-sentence truncation is unacceptable regardless of how comprehensive it otherwise appears. Reject such postings even if they are longer or more detailed than the alternative.
3. Structure and completeness: the posting must include all four required sections '## Summary', '## Required Skills', '## Responsibilities' and '## Requirements', each with substantive, non-empty content. A posting that is missing any of these sections is materially worse than one that includes all four, even if its present sections are richer.
4. Professional and inclusive language: clear, neutral, free of biased or discriminatory phrasing.

Important: length is not a quality indicator. A shorter posting that covers all required content cleanly is equal to or better than a longer posting with similar content. Do not prefer the longer posting unless its extra length is substantive new content, not filler, repetition, or off-topic detail.

You may briefly note the strengths and weaknesses of each candidate before deciding.

End your response with the single line:

VERDICT: A

or:

VERDICT: B

Do not write anything after that line.
```

Position-bias control is the central methodological move. Each pair on each prompt is judged twice with the candidates in opposite orders. Only when the two judgements agree is the pair-prompt counted as a clean win. Disagreement is reported as inconsistency and excluded from the win-rate numerator, but the inconsistency rate is reported separately as a diagnostic for how noisy the judge is.

Outputs go to `data/processed/ma2/eval_winrate.json`. The cell is idempotent on the (pair, prompt, order) key set, so re-running picks up where it left off.


In [42]:
# Section 6.4 - define run_winrate(): pairwise tournament with order-swap
# control, via the atlm_eval_judge agent on agent_server. The agent
# supplies the system prompt, temperature, max_tokens, and
# chat_template_kwargs.
import json
import queue
import re
import threading
import time
import urllib.error
import urllib.request
from itertools import combinations
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR     = PROJECT_ROOT / "data" / "processed" / "ma2"
GEN_DIR     = MA2_DIR / "eval_generations"
EVAL_FILE   = MA2_DIR / "eval_prompts.jsonl"
WINRATE_LOG = MA2_DIR / "winrate_calls.jsonl"
WINRATE_OUT = MA2_DIR / "eval_winrate.json"

JUDGE_MODEL  = "granite-3.3"
JUDGE_AGENT  = "atlm_eval_judge"
TIMEOUT      = 600
WORKERS      = 4

TAGS = ["base", "sft", "dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"]


_VERDICT_RE = re.compile(r"VERDICT:\s*([AB])", re.I)


def _build_judge_user(query: str, a: str, b: str) -> str:
    return (
        f"RECRUITER REQUEST:\n{query}\n\n---\n\n"
        f"CANDIDATE A:\n{a}\n\n---\n\n"
        f"CANDIDATE B:\n{b}\n"
    )


def _call_judge(query: str, a: str, b: str) -> str:
    """Call the atlm_eval_judge agent. Per how_to.md Option A1, just
    {model, messages} - the agent preset supplies everything else."""
    payload = {
        "model": JUDGE_AGENT,
        "messages": [
            {"role": "user", "content": _build_judge_user(query, a, b)},
        ],
    }
    req = urllib.request.Request(
        ENDPOINT, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        resp = json.load(r)
    return resp["choices"][0]["message"]["content"]


def _parse_verdict(text: str):
    m = _VERDICT_RE.search(text or "")
    if not m:
        return None
    return m.group(1).upper()


def _load_done(path):
    if not path.exists():
        return set()
    out = set()
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        out.add((r["pair"], r["prompt_id"], r["order"]))
    return out


def _load_generations():
    by_model = {}
    for tag in TAGS:
        rows = [json.loads(l) for l in open(GEN_DIR / f"{tag}.jsonl", encoding="utf-8")]
        by_model[tag] = {r["prompt_id"]: r["generation"] for r in rows}
    return by_model


def run_winrate():
    # Make sure agent_server has Granite resident before calling the agent.
    switch_active_model(JUDGE_MODEL)

    by_model = _load_generations()
    prompts  = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8")]
    pairs    = list(combinations(TAGS, 2))
    print(f"models: {len(TAGS)}, pairs: {len(pairs)}, prompts: {len(prompts)}")

    tasks = []
    for (a, b) in pairs:
        pair_id = f"{a}_vs_{b}"
        for p in prompts:
            tasks.append((pair_id, a, b, p["id"], p["query"], "ab"))
            tasks.append((pair_id, a, b, p["id"], p["query"], "ba"))

    done = _load_done(WINRATE_LOG)
    todo = [t for t in tasks if (t[0], t[3], t[5]) not in done]
    print(f"calls total {len(tasks)} | already done {len(done)} | todo {len(todo)}")
    if not todo:
        print("nothing to do.")
        return WINRATE_LOG

    q = queue.Queue()
    for t in todo:
        q.put(t)

    f = open(WINRATE_LOG, "a", encoding="utf-8")
    lock = threading.Lock()
    stats = {"ok": 0, "parse_fail": 0, "err": 0}
    t0 = time.time()

    def worker():
        while True:
            try:
                pair_id, a, b, pid, query, order = q.get_nowait()
            except queue.Empty:
                return
            text_a = by_model[a][pid]
            text_b = by_model[b][pid]
            if order == "ab":
                first, second = text_a, text_b
            else:
                first, second = text_b, text_a
            try:
                raw = _call_judge(query, first, second)
            except Exception:
                with lock:
                    stats["err"] += 1
                continue
            v = _parse_verdict(raw)
            with lock:
                if v is None:
                    stats["parse_fail"] += 1
                    continue
                if order == "ab":
                    winner = a if v == "A" else b
                else:
                    winner = b if v == "A" else a
                f.write(json.dumps({
                    "pair":      pair_id,
                    "a":         a, "b": b,
                    "prompt_id": pid, "order": order,
                    "verdict":   v, "winner": winner,
                    "raw":       raw,
                }, ensure_ascii=False) + "\n")
                f.flush()
                stats["ok"] += 1
                n_seen = stats["ok"] + stats["parse_fail"] + stats["err"]
                if n_seen % 20 == 0:
                    el = time.time() - t0
                    print(f"  ok={stats['ok']} parse_fail={stats['parse_fail']} "
                          f"err={stats['err']} | {el/60:.1f} min")

    threads = [threading.Thread(target=worker, daemon=True) for _ in range(WORKERS)]
    for t in threads: t.start()
    for t in threads: t.join()
    f.close()
    el = time.time() - t0
    print(f"done in {el/60:.1f} min | ok={stats['ok']} "
          f"parse_fail={stats['parse_fail']} err={stats['err']}")


def aggregate_winrate():
    """Read the per-call log and produce the per-pair aggregate matrix."""
    rows = [json.loads(l) for l in open(WINRATE_LOG, encoding="utf-8")]
    by_key = {}
    for r in rows:
        by_key.setdefault((r["pair"], r["prompt_id"]), {})[r["order"]] = r["winner"]

    pair_results = {}
    for (pair, pid), verdicts in by_key.items():
        if len(verdicts) < 2:
            continue
        w_ab = verdicts.get("ab")
        w_ba = verdicts.get("ba")
        a, b = pair.split("_vs_")
        pair_results.setdefault(pair, {"a": a, "b": b, "wins_a": 0,
                                       "wins_b": 0, "inconsistent": 0,
                                       "total": 0})
        d = pair_results[pair]
        d["total"] += 1
        if w_ab == w_ba and w_ab is not None:
            if w_ab == a:
                d["wins_a"] += 1
            elif w_ab == b:
                d["wins_b"] += 1
        else:
            d["inconsistent"] += 1

    for d in pair_results.values():
        consistent = d["wins_a"] + d["wins_b"]
        d["winrate_a"] = round(d["wins_a"] / consistent, 3) if consistent else None
        d["winrate_b"] = round(d["wins_b"] / consistent, 3) if consistent else None
        d["agreement_rate"] = round(consistent / d["total"], 3) if d["total"] else None

    WINRATE_OUT.write_text(json.dumps(pair_results, indent=1))
    print(f"\nWrote {WINRATE_OUT.relative_to(PROJECT_ROOT)}")
    print("\n--- pair-by-pair win rates ---")
    print(f"  {'pair':24s} {'wins_a':>6s} {'wins_b':>6s} {'incons':>6s} "
          f"{'wr(a)':>6s} {'wr(b)':>6s} {'agree':>6s}")
    for pair, d in pair_results.items():
        wr_a = "n/a" if d["winrate_a"] is None else f"{d['winrate_a']:.2f}"
        wr_b = "n/a" if d["winrate_b"] is None else f"{d['winrate_b']:.2f}"
        ag   = "n/a" if d["agreement_rate"] is None else f"{d['agreement_rate']:.2f}"
        print(f"  {pair:24s} {d['wins_a']:>6d} {d['wins_b']:>6d} "
              f"{d['inconsistent']:>6d} {wr_a:>6s} {wr_b:>6s} {ag:>6s}")


In [43]:
# Run the win-rate tournament. Requires agent_server up on port 7701.
# ~15-25 min for 240 calls at concurrency 4; idempotent on resume.
run_winrate()
aggregate_winrate()


  [switch] granite-3.3 already active; no-op
models: 6, pairs: 15, prompts: 20
calls total 600 | already done 400 | todo 200
  ok=20 parse_fail=0 err=0 | 0.5 min
  ok=40 parse_fail=0 err=0 | 1.2 min
  ok=60 parse_fail=0 err=0 | 1.9 min
  ok=80 parse_fail=0 err=0 | 2.6 min
  ok=100 parse_fail=0 err=0 | 3.4 min
  ok=120 parse_fail=0 err=0 | 4.2 min
  ok=140 parse_fail=0 err=0 | 4.7 min
  ok=160 parse_fail=0 err=0 | 5.5 min
  ok=180 parse_fail=0 err=0 | 6.3 min
  ok=200 parse_fail=0 err=0 | 7.0 min
done in 7.0 min | ok=200 parse_fail=0 err=0

Wrote data/processed/ma2/eval_winrate.json

--- pair-by-pair win rates ---
  pair                     wins_a wins_b incons  wr(a)  wr(b)  agree
  base_vs_sft                   0      7     13   0.00   1.00   0.35
  base_vs_dpo-b01               0     18      2   0.00   1.00   0.90
  base_vs_dpo-b02               1     14      5   0.07   0.93   0.75
  base_vs_dpo-b03               1     12      7   0.08   0.92   0.65
  sft_vs_dpo-b01                0 

## 6.5 Qualitative side-by-side comparison

The win-rate gives an aggregated picture. The qualitative pass complements it by showing the six models' generations side by side on a small hand-picked set of prompts, chosen to be diagnostic: two from the in-distribution sub-set and two from out-of-distribution. The aim is to surface behavioural differences the win-rate compresses away (verbosity, lost domain knowledge, structural drift, off-prompt content) and to give Section 7 concrete textual evidence.

All six generations use the same decoder kwargs as Section 6.2 (greedy with `repetition_penalty=1.3`), so the qualitative comparison reflects the deployed inference setup exactly.


In [47]:
# Section 6.5 - load the per-model generations and display side-by-side
# for a curated subset of prompts. CPU only.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

GEN_DIR = PROJECT_ROOT / "data" / "processed" / "ma2" / "eval_generations"
TAGS = ["base", "sft", "dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"]

# Four prompts selected for the qualitative pass: two in-distribution and
# two out-of-distribution. Adjust this list if a different subset reads more
# clearly when the run is done.
QUALITATIVE_IDS = ["ind-01", "ind-04", "ood-01", "ood-03"]

# Load per-model generations.
gens = {}
for tag in TAGS:
    rows = [json.loads(l) for l in open(GEN_DIR / f"{tag}.jsonl", encoding="utf-8")]
    gens[tag] = {r["prompt_id"]: r for r in rows}

for pid in QUALITATIVE_IDS:
    sample = gens["base"].get(pid)
    if sample is None:
        print(f"[missing {pid}]")
        continue
    print("=" * 80)
    print(f"PROMPT ID    : {pid}  ({sample['subset']})")
    print(f"REQUEST      : {sample['query']}")
    print("=" * 80)
    for tag in TAGS:
        print(f"\n--- {tag} ---")
        print(gens[tag][pid]["generation"])
    print()


PROMPT ID    : ind-01  (in_distribution)
REQUEST      : We need a backend engineer to build and maintain Python services on AWS, with hands-on experience in Django and PostgreSQL.

--- base ---
* A full description of the role is available here: https://github.com/davidhughes1980/python_jobs#postings



--- sft ---
# Backend Engineer (Python) - Cloud Services

## Summary
We are seeking an experienced Front End Developer specializing in the Python programming language who is proficient in cloud infrastructure technologies like Amazon Web Service or Google Compute Engine. This role involves building robust backends for various applications using modern web frameworks such as Flask/Django. The ideal candidate will be capable of working within Agile environments while maintaining high standards across code quality and performance metrics.

## Required Skills
- Proficiency in at least one version control system including Git
- Experience developing software components based upon object orie

## 6.6 Behavioural comparison across the six model states

Simple summary statistics across the six model states surface the gross behavioural differences the pairwise win-rate in Section 6.4 is responding to.

The metrics are deliberately simple and surface-level: mean output length in characters and words, the count of required Markdown headings actually present in each generation, and side-by-side display of the prompts chosen for maximal behavioural divergence between two contrasting beta values. They surface gross behavioural differences (length, structure adherence, the occasional repetition failure) rather than make fine-grained linguistic judgments; the root-cause analysis lives in Section 7.

Structural completeness varies across models. SFT and DPO-b03 produce all four required Markdown headings on 3 of 20 generations; DPO-b01 on 1 of 20; DPO-b02 and DPO-b005 on 0 of 20. Average required-heading counts per generation are 2.80 (SFT), 2.85 (b03), 2.50 (b01), 2.45 (b02), 1.35 (b005). The shortfall is the `repetition_penalty=1.3` trade-off: restoring the kwarg (Section 6.2 and Section 7.3.1) eliminates the token-loop catastrophes but occasionally triggers an early EOS before the model reaches `## Requirements`. The per-model numbers come from the cell output below.


In [45]:
# Section 6.6 - quantitative behavioural comparison across all five model
# states (base, sft, dpo-b01, dpo-b02, dpo-b03), focused on the b01-vs-b03
# diagnostic deep-dive. CPU only.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

GEN_DIR = PROJECT_ROOT / "data" / "processed" / "ma2" / "eval_generations"
TAGS = ["base", "sft", "dpo-b005", "dpo-b01", "dpo-b02", "dpo-b03"]
REQUIRED = ["## Summary", "## Required Skills",
            "## Responsibilities", "## Requirements"]

def analyze(text: str) -> dict:
    return {
        "chars": len(text),
        "words": len(text.split()),
        "sections_present": sum(1 for s in REQUIRED if s in text),
        "all_4_sections":  all(s in text for s in REQUIRED),
    }

# Load all per-model generations into nested dicts.
gens = {}
for tag in TAGS:
    rows = [json.loads(l) for l in open(GEN_DIR / f"{tag}.jsonl", encoding="utf-8")]
    gens[tag] = {r["prompt_id"]: r for r in rows}

# ---- Per-model aggregate behavioural statistics ----
print("=" * 78)
print("Per-model behavioural statistics across all 20 evaluation prompts")
print("=" * 78)
print(f"  {'model':10s} {'mean_chars':>12s} {'mean_words':>12s}"
      f" {'all_4_sec':>12s} {'avg_secs':>10s}")
print(f"  {'-'*10} {'-'*12} {'-'*12} {'-'*12} {'-'*10}")
for tag in TAGS:
    stats = [analyze(g["generation"]) for g in gens[tag].values()]
    n = len(stats)
    avg_chars = sum(s["chars"] for s in stats) / n
    avg_words = sum(s["words"] for s in stats) / n
    all_4     = sum(s["all_4_sections"] for s in stats)
    avg_secs  = sum(s["sections_present"] for s in stats) / n
    print(f"  {tag:10s} {avg_chars:>12.0f} {avg_words:>12.0f}"
          f"  {f'{all_4}/{n}':>11s} {avg_secs:>10.2f}")

# ---- Section-by-section presence breakdown ----
print()
print("Required-heading presence by model "
      "(number of the 20 generations containing each):")
print(f"  {'section':25s}  " + " ".join(f"{t:>9s}" for t in TAGS))
print(f"  {'-'*25}  " + " ".join("-" * 9 for _ in TAGS))
for sec in REQUIRED:
    counts = [sum(1 for g in gens[tag].values() if sec in g["generation"])
              for tag in TAGS]
    print(f"  {sec:25s}  " + " ".join(f"{c:>9d}" for c in counts))

# ---- Three prompts with the largest DPO-b01 vs DPO-b03 divergence ----
print()
print("=" * 78)
print("Three prompts where DPO-b01 and DPO-b03 differ most (by char count)")
print("=" * 78)
diffs = []
for pid, row_a in gens["dpo-b01"].items():
    row_b = gens["dpo-b03"].get(pid)
    if row_b is None:
        continue
    a, b = row_a["generation"], row_b["generation"]
    diffs.append((pid, abs(len(a) - len(b)), row_a))
diffs.sort(key=lambda x: -x[1])

for i, (pid, dchars, sample) in enumerate(diffs[:3], 1):
    a = gens["dpo-b01"][pid]["generation"]
    b = gens["dpo-b03"][pid]["generation"]
    print()
    print(f"[{i}] prompt_id = {pid}   subset = {sample['subset']}   "
          f"char diff = {dchars}")
    print(f"    REQUEST: {sample['query']}")
    print()
    print(f"--- DPO-b01 ({len(a)} chars, {len(a.split())} words, "
          f"{sum(1 for s in REQUIRED if s in a)}/4 required sections) ---")
    print(a)
    print()
    print(f"--- DPO-b03 ({len(b)} chars, {len(b.split())} words, "
          f"{sum(1 for s in REQUIRED if s in b)}/4 required sections) ---")
    print(b)


Per-model behavioural statistics across all 20 evaluation prompts
  model        mean_chars   mean_words    all_4_sec   avg_secs
  ---------- ------------ ------------ ------------ ----------
  base                694           89         0/20       0.00
  sft                1567          208         3/20       2.80
  dpo-b005            869          119         0/20       1.35
  dpo-b01            1458          196         1/20       2.50
  dpo-b02            1622          216         0/20       2.45
  dpo-b03            1592          211         3/20       2.85

Required-heading presence by model (number of the 20 generations containing each):
  section                         base       sft  dpo-b005   dpo-b01   dpo-b02   dpo-b03
  -------------------------  --------- --------- --------- --------- --------- ---------
  ## Summary                         0        20        13        20        20        20
  ## Required Skills                 0        20        12        20        20 

## 6.7 Synthesis of evaluation results

Three pieces of evidence (Section 6.3 perplexity, Section 6.4 pairwise win-rate, Section 6.6 behavioural statistics) describe what the post-training pipeline produced across the six model states. The synthesis below summarises them: where they agree, where they diverge, and which model emerges as the better-aligned one on which criterion.

The training-side ranking from Section 5.6 (reward margin and accuracy on the held-out preference set) and the deployment-side ranking from Section 6.4 (pairwise win-rate on the 20 frozen prompts) measure different things. This section makes the contrast explicit; Section 7 reads what the contrast means.

Across the three pieces of evidence:

- Perplexity ranks the models in a clean order with respect to the SFT distribution: base (11.66) is far higher than every aligned variant; SFT (4.54) is the lowest. The DPO models rise as beta decreases (4.80, 5.03, 6.01 across b03, b02, b01) and then sharply at b005 (8.84). The sharp jump between b01 (32 percent above SFT) and b005 (95 percent above SFT) is the first signal that beta = 0.05 is past the point where alignment improvement compensates for SFT-distribution divergence.
- Win-rate ranks the DPO models above the base and the SFT model when Granite commits consistently. Aligned models defeat base in 65 to 90 percent of order-swap-consistent verdicts. On the close pairs (SFT vs DPO, DPO vs DPO) Granite is consistent in only 15 to 40 percent of cases; when it is consistent, DPO sweeps SFT 18-0 across the three in-range betas, and the close DPO-vs-DPO verdicts sort lower-beta-preferred. The b005 leg is the only DPO model where SFT consistently wins any prompts (2 of 20), pointing to the same beta = 0.05 cliff that perplexity flagged.
- Behavioural stats add concrete divergence numbers. Structural completeness is uneven across the six model states (SFT 3/20 all-four-section generations, b03 3/20, b01 1/20, b02 0/20, b005 0/20), reflecting the `repetition_penalty=1.3` trade-off documented in Section 7.3.1: the kwarg eliminates the token-loop catastrophes but occasionally triggers an early EOS before the model reaches `## Requirements`. Mean output length is roughly constant across b01, b02 and b03; b005 outputs are about 40 percent shorter, reflecting the entropy collapse that b005 shows on the training-side metric in Section 5.6.

The three pieces agree on the headline. Across the in-range betas (0.10, 0.20, 0.30) the win-rate signal sorts lower-beta-preferred and the perplexity rise is proportionate to the alignment gain. Below that range, at beta = 0.05, the policy diverges aggressively from the SFT distribution without a corresponding lift in judge preference, and on at least 2 of 20 deployment prompts becomes worse than SFT. The deliverable policy is DPO-b01.


# 7. Limitations and critical discussion

Alignment frequently regresses capabilities: the model may become more verbose, refuse more often, lose domain knowledge from earlier stages, or develop sycophancy. This section reports where the alignment helped, where it hurt, and what we would do differently. Honest reporting of these regressions is essential, not optional, since they are the trade-off the technique exposes.

Three methodological choices structure the discussion below. First, this project uses a cross-judge protocol: three distinct external LLMs occupy the data-preparation, RLAIF judging, and evaluation judging roles, so that no single model is both a data source and a verdict source. The motivation, the calibration battery, and the production-scale ceiling that the calibration did not surface are the subject of Section 7.3. Second, the DPO sensitivity probe runs at four beta values rather than the conventional three, with the fourth point (beta = 0.05) added specifically to test whether "lower beta is always better" extrapolates below the conventional 0.1 floor; Section 7.2 carries that result. Third, the post-execution discovery of a missing decoder kwarg carried over unnoticed from Mini-Assignment 1 to Mini-Assignment 2 produced the largest single intervention on the evaluation results; Section 7.3 documents it as both a finding and a methodological lesson.


## 7.1 What alignment changed

The six-way evaluation makes three things clear about the post-training pipeline.

The base SmolLM2-360M model never produces a structured posting. Across all 20 evaluation prompts it loses cleanly to every other model. In Section 6.4 the base versus DPO-b01 pair returns 18 of 20 prompts to DPO-b01 with 90 percent AB-BA agreement (2 inconsistent, zero base wins). The base versus DPO-b02 and base versus DPO-b03 pairs are similar (14-1 and 12-1 in consistent verdicts). Continued pretraining alone, without supervised fine-tuning, does not produce instruction-following behaviour, and the cross-judge protocol consistently identifies the base as the weakest of the six model states.

The supervised fine-tuned model adds the format-following capability. The four required Markdown headings appear on the great majority of supervised generations, and the perplexity on the supervised validation set (4.54) is the lowest of any model in the comparison. The supervised stage produces the posting-shaped outputs the rest of the pipeline operates on; DPO refines those outputs rather than constructing them from scratch.

The DPO models beat the supervised fine-tuned model on the close-pair win-rate whenever the judge commits consistently. The three in-range betas (0.10, 0.20, 0.30) sweep SFT 18 to 0 across the three SFT-vs-DPO pairs in consistent verdicts (8-0, 6-0, 4-0). Zero of those consistent verdicts go to SFT. The b005 leg breaks this pattern with 7 DPO wins versus 2 SFT wins, the only DPO leg where SFT consistently beats any DPO prompts. The DPO stage produced policies the judge consistently prefers over SFT on the in-range betas. Structural completeness, however, lands within a narrow band of SFT but generally lower than the 4/20 all-four-section count one would want for production: 3/20 (SFT), 3/20 (b03), 1/20 (b01), 0/20 (b02), 0/20 (b005). The shortfall is the known cost of the `repetition_penalty=1.3` kwarg restoration documented in Section 7.3.1: it eliminates the token-loop catastrophes at the price of occasional early EOS before `## Requirements`. Section 6.6 confirms the same pattern from the behavioural-statistics angle.

On the prompts where Granite is inconsistent across the order-swap (the majority of close-pair comparisons), the model preference is below the judge's discrimination floor at this judge's parameter budget. Section 7.3 develops what that means.


## 7.2 Beta sensitivity in this run

The four DPO models differ only in beta (0.05, 0.10, 0.20, 0.30); all other hyperparameters are held constant. The original sweep planned three betas (0.10, 0.20, 0.30); the fourth point at 0.05 was added specifically as a cliff probe after the three-point sweep returned a monotonic lower-beta-is-better result that could not on its own distinguish a true monotonic trend from a peak somewhere in the tested range.

Across the in-range three betas the picture is consistent. The DPO-vs-DPO close pairs sort lower-beta-preferred in consistent verdicts (b01 over b02 = 3-0, b01 over b03 = 5-1, b02 over b03 = 2-0, all at 15 to 30 percent AB-BA agreement). The SFT-vs-DPO consistent verdicts sort the same way (8-0, 6-0, 4-0). Eval reward accuracy on training-distribution preferences also peaks at beta = 0.10 (0.715, against 0.694 at both 0.20 and 0.30). Both the training-side metric and the deployment-side metric agree on the lower-is-better direction across the in-range betas.

The b005 cliff probe breaks the pattern. Perplexity on the supervised validation set jumps to 8.84 against 6.01 at b01 (a 47 percent rise) and 4.80 at b03 (a smooth 7 percent rise from SFT's 4.54). Mean output length drops to 869 characters against 1,458 at b01, a 40 percent shorter generation on average. The b005 leg is the only one where SFT consistently wins any prompts (2 of 20). In the DPO-vs-DPO close pairs, b005-vs-b01 is essentially tied (2-1 b005, 17 inconsistent, 15 percent agreement). b005 does beat b02 (5-2) and b03 (5-1) but the win-rate margin no longer scales with the perplexity gap the way it does between b03 and b01.

The interpretation: alignment quality is not monotonic in 1/beta across the tested range. It improves smoothly as beta drops from 0.30 to 0.10 (the alignment-fluency trade-off operating as expected), then plateaus or regresses below 0.10 because over-divergence from the SFT distribution begins to outweigh the alignment signal. The alignment-quality peak sits at beta = 0.10. The deliverable policy is DPO-b01.

The learning-rate history is the second concrete data point in this sensitivity discussion. Three LR values were tried at fixed beta in the iteration documented in Section 5.6: 5e-6 (the textbook DPO value) produced near-zero reward margins; 5e-5 (the deliverable) produced clean positive margins and the beta-axis story above; 1e-4 produced a higher training-side reward margin than 5e-5 but visible fluency degradation, including a learned hallucinated phrase appearing across multiple unrelated prompts at beta = 0.1. The pair (beta = 0.10, learning rate = 5e-5) is the hyperparameter answer this work supports for LoRA-DPO on SmolLM2-360M with the preference dataset used here.

This is a one-dimensional sensitivity probe with four points along beta and three points along learning rate, not a full (beta x LR) grid sweep with replication. A real sweep would map an interaction surface and quantify the uncertainty at each grid point. Section 7.3 acknowledges this in the future-work bullets.


## 7.3 Failure modes, limitations, and future work

The substantive methodological chapter. Every finding below is grounded in a specific file or measurement on disk after Section 6 completes; none of it is asserted from first principles.

### 7.3.1 Decoder-loop attractor and the Mini-Assignment 1 inference-helper regression

The first execution of Section 6 produced catastrophic token-level repetition collapses on some DPO models. The clearest case was DPO-b02 on the cloud-engineer prompt `ind-07`, where the Required Skills section contained 42 consecutive repetitions of the token `SQS` as bullet items. DPO-b03 on the same prompt produced 29 repetitions of `ECS ECS ECS`. DPO-b01 had milder but broader-spread repetition across `ind-07`, `ind-08` and `ind-10`. Section 6.4 win-rate computed on these generations floored at 5 to 15 percent AB-BA agreement on close pairs, below random and consistent with strong systematic position bias in Granite's verdicts.

Two diagnostic findings followed that the training-side metrics could not have produced. First, the failure was not inherited from training data. A scan of `data/processed/converted.jsonl` (the 2,507-record supervised fine-tuning corpus) found the phrase "Container orchestration using" exactly 7 times, with a maximum of 1 occurrence per record. The SFT corpus contains no example with a repetitive list of this shape. The failure is emergent at inference, not memorised from training. Second, the Mini-Assignment 1 inference helper `src/generate_mp1.py` carried `repetition_penalty=1.3` explicitly with the comment "small continued-pretrained models loop easily". The Mini-Assignment 2 evaluation generation function in `ma2s62code` dropped this kwarg, silently, across versions v3 through v6. Restoring `repetition_penalty=1.3` to the v6 evaluation generation eliminated all token-loop catastrophes across the 120 outputs (six models times 20 prompts), with zero prompts showing severe repetition in the post-fix run.

Re-running Section 6.4 against the clean generations moved the close-pair AB-BA agreement from 5-15 percent up to 15-40 percent, off the position-bias floor. When Granite committed on close pairs after the fix, it picked DPO over SFT consistently (18 close-pair consistent verdicts, all DPO wins, across the three in-range SFT-vs-DPO pairs) and produced the b01 > b02 > b03 ordering documented in Section 7.2. The catastrophic outputs in the first execution had been actively confounding Granite's verdicts; with them removed, the real DPO signal partially surfaced.

The methodological lesson is that inference-time decoder kwargs are part of the system, not an afterthought. The same `repetition_penalty` value that solved this failure mode on the same model family in Mini-Assignment 1 was a load-bearing dependency in Mini-Assignment 2, and dropping it produced apparently methodological-grade failures (mode collapse) that the training-side metrics could not see. Cross-stage inference-kwarg parity is now a delivery checklist item in this team.

### 7.3.2 Eval judge discrimination ceiling

Granite 3.3 2B was selected from the calibration battery in [`documents/development/llm_models_performance.md`](../../documents/development/llm_models_performance.md) on the strength of a 5-pair x 2-order eval probe (5 of 5 parse, 4 of 5 AB-BA consistent on the comparable pairs, caught the deliberately-broken `ood-03` collapse, fast). At production scale on 600 calls (15 pairs x 20 prompts x 2 orders), close-pair AB-BA agreement floored at 5-15 percent in the first execution (with the catastrophic outputs) and at 15-40 percent in the second (with the clean outputs). On easy pairs (base versus any aligned model) the same judge produces 65-90 percent agreement. The cross-judge protocol surfaced a structural discrimination ceiling on close pairs that the calibration battery did not.

The math is informative. If Granite were judging randomly, AB-BA agreement would be 50 percent (independent coin flips). If Granite were purely position-biased (always picking whichever candidate is in a fixed position), AB-BA agreement would be 0 percent (AB picks the position-X candidate; BA picks the same position-X candidate, which is now the other model). The 5-15 percent close-pair regime is below random and close to the pure-position-bias floor; the 15-40 percent regime is between random and the easy-pair regime, consistent with partial signal recovery once the decoder bug was fixed. On the prompts where Granite did commit, inspection of the raw verdicts showed it occasionally names a candidate's weakness (a repetition loop, say) in the reasoning text and then votes for that candidate anyway. The rubric is in the system prompt, but applying it reliably to close-pair comparisons is at the edge of what a 2B-parameter judge can do on this task.

The methodological lesson is that the calibration sample size was undersized for the question the calibration was asked. A 10-call probe is statistically incapable of distinguishing 80 percent AB-BA from 50 percent AB-BA at any meaningful confidence; production scale at 600 calls is what surfaced the floor. A close-pair-only calibration with larger N would have flagged the issue before the pipeline was built on it.

### 7.3.3 DPO coverage bound

DPO can only suppress failure modes that appear in its candidate distribution. The `ind-07` SFT defect that initiated the decoder-loop investigation (a tendency to fill the Required Skills section with "Container orchestration using X" bullets) is outside the 250 preference prompts that Section 4 draws; DPO had no gradient pointing at this specific failure mode, which is why the failure persisted across all four DPO variants and surfaced as the catastrophic token loops in the first execution. This is a concrete instance of a theoretical bound: DPO refines the policy in the regions where the candidate distribution and the judge's preferences both have signal. Outside those regions, DPO neither helps nor hurts; the underlying SFT behaviour passes through unchanged.

### 7.3.4 Evaluation set size

The 20-prompt evaluation set was chosen for tractability against a cross-judge protocol with order-swap, agent_server preset infrastructure and active-model switching overhead. The size is sufficient to surface the headline base-versus-aligned gap and to detect cross-beta directional differences when Granite commits, but the binomial confidence intervals on close-pair comparisons are wide. An 8-0 consistent DPO sweep against SFT is robust at N=20; a 3-0 close-pair count between two DPO models is at the edge of significance and is reported as suggestive rather than conclusive. Section 7.1's headline claims (DPO improves over SFT on close pairs; b01 is the deliverable) are at the robust end; Section 7.2's b005-vs-b01 essentially-tied conclusion (2-1, 17 inconsistent) is at the noisy end.

### 7.3.5 Preference-data volume was not the bottleneck

The v6 pivot to full-ranking judging produced 1,446 preference triples, a 6.5x scaling versus the 222 best-worst triples used in Configurations B (v4_AC) and C (v5_AC). The scaling did not move the over-fit signature (train_rwd_acc reached 1.00 across all four betas in v6, equivalent to Configurations B and C's near-saturation), did not move the close-pair judge discrimination floor (Configurations C and v6 sit at the same 5-15 percent or 15-40 percent regimes depending on output cleanness), and did not produce a clear deployment-quality lift over Configuration C's three-beta result. Preference volume was not the right lever for this configuration. The bottleneck was preference signal noise plus judge ceiling.

### 7.3.6 Iteration history as itself a finding

The Configuration A (Gemma judge, first end-to-end, undersized `max_tokens` caused 99 percent parse failure) to B (Gemma fixed, headline result "DPO-b01 wins decisively" grounded in Gemma's self-preferences) to C (cross-judge protocol, surfaced Granite's close-pair floor and identified that the Configuration B `ood-03` collapse was Gemma-judge-specific rather than low-beta-intrinsic) to v6 (consolidated four-beta, full-ranking, post-execution decoder-bug fix) trajectory is itself a substantive methodological story. Each iteration's failure motivated the next iteration's design. The artefacts of every configuration are preserved on disk under suffixed output directories so the trajectory is reproducible.

### 7.3.7 Future work

Prioritised by expected impact, given more compute and time:

- A larger evaluation judge. Either Gemma 4 (which the v4_AC run already showed produces decisive close-pair verdicts at this model scale, at the cost of weakening cross-family independence with the ETL teacher role), a Qwen3.5-9B-class model (subject to recalibration; the smaller Qwen variant on our agent_server failed the calibration battery on position bias and parse rate), or a judge ensemble. The Granite ceiling is the largest single bound on what this evaluation could measure.
- A larger N evaluation set, ideally on the order of 200 stratified prompts across the same in-distribution / out-of-distribution split. At N=200, the close-pair binomial confidence intervals would shrink enough to make 3-0 or 5-1 close-pair counts statistically interpretable rather than suggestive.
- A full (beta x LR) sensitivity grid with replication, against the same eval surface. The four-point beta probe at fixed LR locates the alignment-quality peak; what it cannot tell us is whether the peak shifts at different learning rates.
- Improving the SFT corpus quality. The `ind-07` SFT defect surfaced by the decoder-loop investigation is one concrete example; the supervised fine-tuning corpus produced by the `atlm_teacher` agent occasionally generates patterned list shapes that the small student model then memorises. Better teacher-model output filtering or a higher-quality teacher would reduce inherited structural defects.
- Increasing candidate diversity in the RLAIF stage. Four candidates from the same SFT model at temperature 0.9 produced limited candidate variance; eight candidates from two different temperatures, or candidates from intermediate SFT checkpoints, would provide more meaningful preference triples where preference volume alone did not.
